# Membership customer profiling — V2 (Parquet export)

**Initial analysis, built without inspecting the underlying data file.** This reuses the analytical
pipeline from `customer_profiling.ipynb`, pointed at a new source —
[`dbprof5_jjscarwash_000042_customer_events.parquet`](dbprof5_jjscarwash_000042_customer_events.parquet)
— described here only by the column dictionary supplied for this build. Every number below is
computed live when this notebook runs; nothing is asserted in advance the way V1's write-up could,
since V1 was written after reviewing real Hurricane-export results and this one is not.

### Column dictionary (as given, not verified against the file)

| Column | Meaning |
|---|---|
| `site_id` | unique identifier for a site |
| `membership_package_name` | membership package name |
| `customer_id` | unique identifier of customer |
| `customer_no` | a second customer identifier — described as "sort of" unique, so **not** used as a key here; `customer_id` remains the key throughout, exactly as in V1 |
| `vehicle_id` | unique identifier of vehicle |
| `vehicle_license` | plate number |
| `vehicle_state` | state the vehicle is registered in |
| `vehicle_make` / `vehicle_model` / `vehicle_year` / `vehicle_color` / `vehicle_vin` | vehicle attributes |
| `vehicle_type` | vehicle category |
| `vehicle_active` | **new field, not in V1** — vehicle active flag |
| `vehicle_black_listed` | **new field, not in V1** — vehicle blacklist flag |
| `event_date` | when the event happened |
| `event_type` | `payment` or `wash` |
| `payment_type` | `signup` vs `renewal` — new vs. recurring customer |
| `amount` | charge amount; populated only when `event_type == payment` |
| `current_package_price` | the package's list price at the time of the charge |

### What's inherited from V1, and what's flagged rather than assumed

Shared logic lives in [`profiling.py`](profiling.py) — the same module V1 and the Streamlit demo
use. Three constants there (`CYCLE_DAYS=30`, `RENEW_WINDOW=45`, `CHURN_AFTER=40` days) and the
"promo" definition (signup amount < 60% of list price) were **empirical fits to the original
Hurricane export**, not structural truths about every membership business. §1 and §4 below check
whether they still look reasonable here and say so explicitly rather than assuming it.

One override on top of that: **this notebook uses `CHURN_AFTER=90` days**, not V1's 40 — a customer
counts as churned here only after 90 days with no payment, not 40. `customer_table()` and
`renewal_panel()` in `profiling.py` both take `churn_after` as a parameter (defaulting to 40, V1's
value, unchanged) specifically so this override doesn't touch V1's or the demo's behavior.

**This notebook restricts every table and chart to `DATE_RANGE_START`-`DATE_RANGE_END` = Jan 2020
through Jul 2026 inclusive.** The filter is applied to `raw` and `ev` right after loading, before
`pay`, `wsh`, `cust`, `panel`, or `seg` are built from them, so it's consistent everywhere rather
than patched into each chart individually. This also covers dropping August 2026 on its own (it
falls after the end of the range), which is why that's no longer a separate filter.

Unlike V1, `load_events()` here is called with `drop_cols=[]` — it does **not** pre-drop
`vehicle_vin`/`vehicle_year`/`vehicle_model`, since nothing has verified those are mostly empty on
this export the way they were on the original one. §2's data-quality audit is a genuine first look,
not a formality.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import Markdown, display
from scipy import stats

sys.path.insert(0, str(Path.cwd()))
import profiling as P
import viz

pio.renderers.default = "notebook"
T = viz.theme(dark=False)          # the notebook renders on white
SEG_COLOR = viz.segment_colors(T)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)


def insight(bullets, title="Insights"):
    # Renders a Reading/So-what/Caveat block from LIVE-computed bullet strings. Unlike V1's static
    # Insights cells (written after reviewing real results), every bullet passed here is built from
    # variables computed earlier in THIS run -- nothing is hard-coded from a prior look at the data.
    display(Markdown(f"**{title}**\n\n" + "\n".join(f"- {b}" for b in bullets)))


DATA_V2 = Path.cwd() / "dbprof5_jjscarwash_000042_customer_events.parquet"
CHURN_AFTER_V2 = 90   # override of profiling.CHURN_AFTER (40, V1's Hurricane fit) -- see intro note
DATE_RANGE_START = pd.Timestamp("2020-01-01")   # inclusive -- every chart/table below is
DATE_RANGE_END = pd.Timestamp("2026-08-01")     # exclusive -- restricted to this window

raw = pd.read_parquet(DATA_V2)
raw["event_date"] = pd.to_datetime(raw["event_date"])
print(f"{len(raw):,} raw rows x {raw.shape[1]} columns")

_raw_before = len(raw)
raw = raw[(raw.event_date >= DATE_RANGE_START) & (raw.event_date < DATE_RANGE_END)].reset_index(drop=True)
print(f"Restricted to {DATE_RANGE_START.date()} - "
      f"{(DATE_RANGE_END - pd.Timedelta(days=1)).date()}: dropped {_raw_before - len(raw):,} rows "
      f"outside that range ({_raw_before:,} -> {len(raw):,} rows) -- kept out of every "
      "chart/table below.")
raw.head(4)

2,151,273 raw rows x 20 columns
Restricted to 2020-01-01 - 2026-07-31: dropped 70,290 rows outside that range (2,151,273 -> 2,080,983 rows) -- kept out of every chart/table below.


,site_id,membership_package_name,customer_id,customer_no,vehicle_id,vehicle_license,vehicle_state,vehicle_make,vehicle_model,vehicle_year,vehicle_color,vehicle_vin,vehicle_type,vehicle_active,vehicle_black_listed,event_date,event_type,payment_type,amount,current_package_price
0,3,Windsong Annual Plan Comp,1,064104174,NaN,NaN,NaN,NaN,NaN,NaN,None,None,NaN,NaN,NaN,2020-01-03 23:41:52,wash,NaN,NaN,NaN
1,3,Windsong Annual Plan Comp,1,064104174,NaN,NaN,NaN,NaN,NaN,NaN,None,None,NaN,NaN,NaN,2020-01-08 15:10:22,wash,NaN,NaN,NaN
2,3,Windsong Annual Plan Comp,1,064104174,124385.0,6DV9761,TX,NaN,NaN,NaN,None,None,NaN,0.0,0.0,2020-01-09 06:05:00,payment,renewal,44.0,NaN
3,3,Windsong Annual Plan Comp,1,064104174,36633.0,RMR4116,TX,Toyota,NaN,NaN,None,None,NaN,0.0,0.0,2020-01-09 06:05:00,payment,renewal,44.0,NaN


In [2]:
raw[raw["customer_id"] == 2257]

,site_id,membership_package_name,customer_id,customer_no,vehicle_id,vehicle_license,vehicle_state,vehicle_make,vehicle_model,vehicle_year,vehicle_color,vehicle_vin,vehicle_type,vehicle_active,vehicle_black_listed,event_date,event_type,payment_type,amount,current_package_price
323913,3,Better Wash Monthly Membership Plan,2257,050306444,1911.0,KXV3301,NaN,Ford,Fiesta,2018.0,None,None,NaN,1.0,0.0,2020-09-13 20:50:14,payment,renewal,13.0,35.0
323914,3,Better Wash Monthly Membership Plan,2257,050306444,1911.0,KXV3301,NaN,Ford,Fiesta,2018.0,None,None,NaN,1.0,0.0,2020-09-13 20:50:14,payment,renewal,-13.0,35.0
323915,3,Better Wash Monthly Membership Plan,2257,050306444,1911.0,KXV3301,NaN,Ford,Fiesta,2018.0,None,None,NaN,1.0,0.0,2020-09-13 20:50:14,payment,renewal,30.0,35.0
323916,1,Better Wash Monthly Membership Plan,2257,050306444,NaN,NaN,NaN,NaN,NaN,NaN,None,None,NaN,NaN,NaN,2020-09-25 14:56:08,wash,NaN,NaN,NaN
323917,1,Better Wash Monthly Membership Plan,2257,050306444,NaN,NaN,NaN,NaN,NaN,NaN,None,None,NaN,NaN,NaN,2020-09-30 17:03:48,wash,NaN,NaN,NaN
323918,1,Better Wash Monthly Membership Plan,2257,050306444,NaN,NaN,NaN,NaN,NaN,NaN,None,None,NaN,NaN,NaN,2020-10-01 18:40:58,wash,NaN,NaN,NaN
323919,1,Better Wash Monthly Membership Plan,2257,050306444,NaN,NaN,NaN,NaN,NaN,NaN,None,None,NaN,NaN,NaN,2020-10-02 15:50:02,wash,NaN,NaN,NaN
323920,3,Better Wash Monthly Membership Plan,2257,050306444,NaN,NaN,NaN,NaN,NaN,NaN,None,None,NaN,NaN,NaN,2020-10-02 18:46:37,wash,NaN,NaN,NaN
323921,1,Better Wash Monthly Membership Plan,2257,050306444,NaN,NaN,NaN,NaN,NaN,NaN,None,None,NaN,NaN,NaN,2020-10-04 18:14:28,wash,NaN,NaN,NaN
323922,1,Better Wash Monthly Membership Plan,2257,050306444,NaN,NaN,NaN,NaN,NaN,NaN,None,None,NaN,NaN,NaN,2020-10-06 19:00:35,wash,NaN,NaN,NaN


In [3]:
raw

,site_id,membership_package_name,customer_id,customer_no,vehicle_id,vehicle_license,vehicle_state,vehicle_make,vehicle_model,vehicle_year,vehicle_color,vehicle_vin,vehicle_type,vehicle_active,vehicle_black_listed,event_date,event_type,payment_type,amount,current_package_price
0,3,Windsong Annual Plan Comp,1,064104174,NaN,NaN,NaN,NaN,NaN,NaN,None,None,NaN,NaN,NaN,2020-01-03 23:41:52,wash,NaN,NaN,NaN
1,3,Windsong Annual Plan Comp,1,064104174,NaN,NaN,NaN,NaN,NaN,NaN,None,None,NaN,NaN,NaN,2020-01-08 15:10:22,wash,NaN,NaN,NaN
2,3,Windsong Annual Plan Comp,1,064104174,124385.0,6DV9761,TX,NaN,NaN,NaN,None,None,NaN,0.0,0.0,2020-01-09 06:05:00,payment,renewal,44.0,NaN
3,3,Windsong Annual Plan Comp,1,064104174,36633.0,RMR4116,TX,Toyota,NaN,NaN,None,None,NaN,0.0,0.0,2020-01-09 06:05:00,payment,renewal,44.0,NaN
4,3,Windsong Annual Plan Comp,1,064104174,61748.0,GT5Y4544H,NaN,NaN,NaN,NaN,None,None,NaN,1.0,0.0,2020-01-09 06:05:00,payment,renewal,44.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2080978,4,Better Wash Monthly Membership Plan,28854,048870866,137771.0,YGC7782,TX,Chevrolet,NaN,NaN,None,None,NaN,1.0,0.0,2026-07-31 22:15:05,payment,signup,17.0,35.0
2080979,1,Good Wash Monthly Membership Plan,28855,075879574,86060.0,VKG4557,TX,Honda,NaN,NaN,None,None,NaN,1.0,0.0,2026-07-31 22:23:33,payment,signup,12.0,30.0
2080980,3,Ceramic Wash Monthly Membership Plan,28856,051347488,126318.0,XBF7924,TX,Ford,NaN,NaN,None,None,Passenger Vehicle,1.0,0.0,2026-07-31 22:30:27,payment,signup,25.0,50.0
2080981,4,Ceramic Wash Monthly Membership Plan,28857,010960408,139520.0,PLN8893,TX,Mazda,,NaN,None,None,NaN,1.0,0.0,2026-07-31 22:38:11,payment,signup,25.0,50.0


---
## 1. Preprocessing — two traps in the file's shape

The export is one row per **(event x vehicle)**. That single fact decides almost every number
downstream, and it cuts in opposite directions for the two event types.

**Trap 1 — payments are fanned out across the household's vehicles.** A multi-car household paying
once can appear as several identical payment rows. Summing `amount` naively counts the charge once
per vehicle on the account, not once per charge.

**Trap 2 — washes are *not* fanned out, and must not be collapsed.** A household with several cars
really does drive several washes. The only legitimate wash duplicates come from one `vehicle_id`
logged under two spellings of its plate.

Getting these backwards — collapsing washes, or not collapsing payments — is the difference between
a correct P&L and a systematically wrong one. Both examples below are found dynamically, not
hand-picked, since no specific `customer_id` from this export is known in advance.

In [4]:
# Trap 1, found dynamically: a household whose payment rows fan out across multiple vehicles
# on the same charge.
_pay_raw = raw[raw.event_type == "payment"]
_fanout = _pay_raw.groupby(["customer_id", "event_date"])["vehicle_id"].transform("nunique")
_candidates = _pay_raw.loc[_fanout.gt(1), "customer_id"]
example_fanout_customer = _candidates.iloc[0] if len(_candidates) else None

if example_fanout_customer is not None:
    display(_pay_raw[_pay_raw.customer_id == example_fanout_customer]
            [["customer_id", "vehicle_id", "vehicle_license", "event_date", "payment_type", "amount"]]
            .head(6))
else:
    print("No fanned-out payment rows found -- every payment already carries exactly one "
          "vehicle_id in this export, so Trap 1 may not apply here.")

,customer_id,vehicle_id,vehicle_license,event_date,payment_type,amount
2,1,124385.0,6DV9761,2020-01-09 06:05:00,renewal,44.0
3,1,36633.0,RMR4116,2020-01-09 06:05:00,renewal,44.0
4,1,61748.0,GT5Y4544H,2020-01-09 06:05:00,renewal,44.0
5,1,93235.0,SNM9944,2020-01-09 06:05:00,renewal,44.0
6,1,64750.0,TJK1265,2020-01-09 06:05:00,renewal,44.0
7,1,93232.0,VKM9189,2020-01-09 06:05:00,renewal,44.0


In [5]:
# Trap 2, found dynamically: a vehicle_id whose wash rows carry two different licence-plate
# spellings at the same timestamp.
_wsh_raw = raw[raw.event_type == "wash"]
_dupe = _wsh_raw.groupby(["customer_id", "vehicle_id", "event_date"])["vehicle_license"].transform("nunique")
_dupe_rows = _wsh_raw.loc[_dupe.gt(1)]

if len(_dupe_rows):
    ex = _dupe_rows.iloc[0]
    display(_wsh_raw[(_wsh_raw.customer_id == ex.customer_id) & (_wsh_raw.vehicle_id == ex.vehicle_id)
                     & (_wsh_raw.event_date == ex.event_date)]
            [["customer_id", "vehicle_id", "vehicle_license", "vehicle_make", "event_date"]])
else:
    print("No same-vehicle, same-timestamp plate-spelling duplicates found in this export.")

,customer_id,vehicle_id,vehicle_license,vehicle_make,event_date
725,1,23551.0,TJK1265,BMW,2023-05-12 16:39:14
726,1,23551.0,IGIX35,Kia,2023-05-12 16:39:14


In [6]:
ev = P.load_events(DATA_V2, drop_cols=[])   # do NOT pre-drop columns -- see intro note

_ev_before = len(ev)
ev = ev[(ev.event_date >= DATE_RANGE_START) & (ev.event_date < DATE_RANGE_END)].reset_index(drop=True)
print(f"Restricted to {DATE_RANGE_START.date()} - "
      f"{(DATE_RANGE_END - pd.Timedelta(days=1)).date()}: dropped {_ev_before - len(ev):,} rows "
      f"outside that range from the cleaned event log ({_ev_before:,} -> {len(ev):,} rows). Every "
      "table/chart from here on -- payments, washes, cohorts, hazard curve, model, segments, "
      "economics -- is built from this filtered `ev`.")

pay = P.payments(ev)
wsh = P.washes(ev)

naive = raw[raw.event_type == "payment"].amount.sum()
pd.DataFrame({
    "rows in export": [(raw.event_type == "payment").sum(), (raw.event_type == "wash").sum()],
    "after handling": [len(pay), len(wsh)],
    "revenue naive":  [f"${naive:,.0f}", ""],
    "revenue correct": [f"${pay.amount.sum():,.0f}", ""],
}, index=["payments", "washes"])

Restricted to 2020-01-01 - 2026-07-31: dropped 70,261 rows outside that range from the cleaned event log (2,150,413 -> 2,080,152 rows). Every table/chart from here on -- payments, washes, cohorts, hazard curve, model, segments, economics -- is built from this filtered `ev`.


,rows in export,after handling,revenue naive,revenue correct
payments,740814,391955,"$36,110,207","$16,559,252"
washes,1340169,1290716,,


In [7]:
naive

np.float64(36110207.14)

In [8]:
_correct_rev = pay.amount.sum()
_inflation_pct = (naive / _correct_rev - 1) * 100 if _correct_rev else float("nan")
_raw_washes = (raw.event_type == "wash").sum()
_wash_dupe_pct = (1 - len(wsh) / _raw_washes) * 100 if _raw_washes else float("nan")
_n_multi_vehicle = int((pay.groupby("customer_id")["n_vehicles_billed"].max() > 1).sum())
_n_customers = ev.customer_id.nunique()

insight([
    (f"**Reading — the fan-out {'inflates' if _inflation_pct >= 0.5 else 'does not meaningfully inflate'} "
     f"revenue** ({_inflation_pct:+.1f}%). Summing raw `amount` gives ${naive:,.0f}; collapsing to "
     f"{len(pay):,} distinct charges gives ${_correct_rev:,.0f}. Every dollar figure from here on "
     "uses the latter."),
    (f"**Reading — the wash side has {_wash_dupe_pct:.1f}% duplicate rows** "
     f"({_raw_washes:,} raw rows -> {len(wsh):,} vehicle-washes)."),
    (f"**So-what — household size is concentrated, not evenly spread.** {_n_multi_vehicle} of "
     f"{_n_customers} customers ({_n_multi_vehicle / max(_n_customers, 1):.1%}) have 2+ vehicles "
     "billed at some point -- if that share is small, the fan-out correction matters disproportionately "
     "for the best customers, same pattern as V1."),
])

**Insights**

- **Reading — the fan-out inflates revenue** (+118.1%). Summing raw `amount` gives $36,110,207; collapsing to 391,955 distinct charges gives $16,559,252. Every dollar figure from here on uses the latter.
- **Reading — the wash side has 3.7% duplicate rows** (1,340,169 raw rows -> 1,290,716 vehicle-washes).
- **So-what — household size is concentrated, not evenly spread.** 4459 of 20277 customers (22.0%) have 2+ vehicles billed at some point -- if that share is small, the fan-out correction matters disproportionately for the best customers, same pattern as V1.

---
## 2. Data-quality audit

What is actually populated, and what has to be handled with care before modelling. Unlike V1, this
table has not been used to decide what to drop — that decision, if any, belongs after seeing it.

In [9]:
q = pd.DataFrame({
    "null %": (raw.isna().mean() * 100).round(1),
    "distinct": raw.nunique(),
}).sort_values("null %", ascending=False)
q["verdict"] = np.where(q["null %"] > 90, "DROP - unusable",
                 np.where(q["null %"] > 15, "partial - use with care", "usable"))
q

,null %,distinct,verdict
vehicle_color,100.0,0,DROP - unusable
vehicle_vin,100.0,0,DROP - unusable
vehicle_type,76.4,6,partial - use with care
vehicle_year,74.4,52,partial - use with care
vehicle_model,65.8,1789,partial - use with care
amount,64.4,115,partial - use with care
payment_type,64.4,2,partial - use with care
current_package_price,64.4,10,partial - use with care
vehicle_state,27.8,58,partial - use with care
vehicle_make,25.8,411,partial - use with care


In [10]:
_drop_verdict = q[q.verdict == "DROP - unusable"].index.tolist()
_new_cols = [c for c in ["customer_no", "vehicle_active", "vehicle_black_listed"] if c in q.index]
_missing_expected = [c for c in ["customer_no", "vehicle_active", "vehicle_black_listed"] if c not in q.index]

bullets = [
    (f"**Reading — {len(_drop_verdict)} column(s) look unusable on this export:** "
     + (", ".join(f"`{c}` ({q.loc[c, 'null %']:.1f}% null)" for c in _drop_verdict) if _drop_verdict
        else "none — every column carries at least some signal, unlike V1 where three vehicle "
             "columns were >=96% null.")),
]
if _new_cols:
    bullets.append("**Reading — fields new to this export:** " + ", ".join(
        f"`{c}` ({q.loc[c, 'null %']:.1f}% null, {int(q.loc[c, 'distinct'])} distinct)" for c in _new_cols))
if _missing_expected:
    bullets.append(f"*Caveat:* the data dictionary mentioned {', '.join(_missing_expected)}, but "
                    "the export doesn't have that column — worth confirming the file matches "
                    "expectations before trusting anything derived from it.")
bullets.append("**So-what — whatever V1 found usable behaviourally (payment amount, wash cadence) "
               "has no guarantee of holding structurally here** until the table above has been "
               "reviewed against it.")
bullets.append("*Caveat:* a `null %` of 0 does not mean a column is well-formed — e.g. `\"-\"` as a "
               "spelling of missing (handled below for `vehicle_state`/`vehicle_type`/`vehicle_make`/"
               "`vehicle_color`, per V1's convention) would show as 0% null here but still needs it.")
insight(bullets)

**Insights**

- **Reading — 2 column(s) look unusable on this export:** `vehicle_color` (100.0% null), `vehicle_vin` (100.0% null)
- **Reading — fields new to this export:** `customer_no` (0.0% null, 20344 distinct), `vehicle_active` (19.3% null, 2 distinct), `vehicle_black_listed` (19.3% null, 2 distinct)
- **So-what — whatever V1 found usable behaviourally (payment amount, wash cadence) has no guarantee of holding structurally here** until the table above has been reviewed against it.
- *Caveat:* a `null %` of 0 does not mean a column is well-formed — e.g. `"-"` as a spelling of missing (handled below for `vehicle_state`/`vehicle_type`/`vehicle_make`/`vehicle_color`, per V1's convention) would show as 0% null here but still needs it.

In [11]:
cust = P.customer_table(ev, churn_after=CHURN_AFTER_V2)
issues = pd.DataFrame([
    ("exact duplicate rows dropped", len(raw) - len(ev), "removed at load"),
    ("customers with washes but no payment row", int(cust.cycles_paid.isna().sum()),
     "excluded from economics"),
    ("charges with no vehicle_id on file", int(pay.no_vehicle_on_file.sum()), "flagged, not imputed"),
    ("$0 renewals (comped months)", int(pay.is_comp.sum()), "kept -- member is still live"),
    ("negative amounts (refunds)", int(pay.is_refund.sum()),
     f"${pay[pay.is_refund].amount.sum():,.0f}, {pay[pay.is_refund].customer_id.nunique()} members"),
], columns=["issue", "count", "handling"])
issues

,issue,count,handling
0,exact duplicate rows dropped,831,removed at load
1,customers with washes but no payment row,605,excluded from economics
2,charges with no vehicle_id on file,13889,"flagged, not imputed"
3,$0 renewals (comped months),3699,kept -- member is still live
4,negative amounts (refunds),12756,"$-275,861, 6741 members"


In [12]:
_n_no_pay = int(cust.cycles_paid.isna().sum())
_n_dupe = len(raw) - len(ev)
insight([
    f"**Reading — {_n_dupe:,} exact duplicate rows were dropped at load** "
    f"({_n_dupe / max(len(raw), 1):.1%} of the export).",
    (f"**{_n_no_pay} customer(s) have washes but no payment row** and are excluded from economics."
     if _n_no_pay else "**No customers have washes without a payment row.**"),
    f"**{int(pay.is_comp.sum())} `$0` renewals** are kept as live members, not treated as errors.",
    (f"**{int(pay.is_refund.sum())} negative-amount rows (refunds)**, "
     f"${pay[pay.is_refund].amount.sum():,.0f} across {pay[pay.is_refund].customer_id.nunique()} "
     "members, netted into revenue but excluded from the cycle count."
     if pay.is_refund.any() else "**No refunds (negative amounts) in this export.**"),
])

**Insights**

- **Reading — 831 exact duplicate rows were dropped at load** (0.0% of the export).
- **605 customer(s) have washes but no payment row** and are excluded from economics.
- **3699 `$0` renewals** are kept as live members, not treated as errors.
- **12756 negative-amount rows (refunds)**, $-275,861 across 6741 members, netted into revenue but excluded from the cycle count.

---
## 3. The book at a glance

Every retention/churn number below only means something in context of how mature this book is —
computed live, next, rather than stated as a known fact the way V1 could after reviewing its export.

In [13]:
asof = P.asof(ev)
_first_payment = pay[~pay.is_refund].event_date.min()
_span_days = (asof - _first_payment).days
print(f"first payment: {_first_payment.date()}   |   export ends: {asof.date()}   |   "
      f"span: {_span_days} days (~{_span_days / 30.44:.1f} months)")

first payment: 2020-01-01   |   export ends: 2026-07-31   |   span: 2403 days (~78.9 months)


In [14]:
joins = cust.groupby("cohort").size().rename("joined")
lost = cust[cust.churned].groupby("churn_month").size().rename("churned")
flow = pd.concat([joins, lost], axis=1).fillna(0).astype(int).sort_index()
flow["net"] = flow.joined - flow.churned
# Cumulative active-member count, computed over the FULL flow (including whatever's in the first
# row) so early cohorts still contribute to the running total even where the bars below don't show
# their join month individually.
flow["active_total"] = flow.net.cumsum()

# The very first month in the export is a one-time bulk-backfill artifact on many exports (the
# whole pre-existing book landing on day one), not organic acquisition -- on this one it dwarfs
# every other month's bar on the same axis. Trim the BAR display to skip it; the active-member LINE
# still reflects its contribution correctly, since active_total is a cumulative sum computed above,
# before this trim.
FLOW_DISPLAY_START = "2020-02"
flow_display = flow.loc[FLOW_DISPLAY_START:]

fig = go.Figure()
fig.add_bar(x=flow_display.index, y=flow_display.joined, name="Joined", marker_color=T.s1,
            hovertemplate="%{x}<br>%{y} joined<extra></extra>")
fig.add_bar(x=flow_display.index, y=-flow_display.churned, name="Churned", marker_color=T.s2,
            hovertemplate="%{x}<br>%{customdata} churned<extra></extra>", customdata=flow_display.churned)
fig.add_scatter(x=flow_display.index, y=flow_display.net, name="Net change", mode="lines+markers",
                line=dict(color=T.ink, width=2), marker=dict(size=8), yaxis="y1",
                hovertemplate="%{x}<br>net %{y:+d}<extra></extra>")
fig.add_scatter(x=flow_display.index, y=flow_display.active_total, name="Total active members",
                mode="lines", line=dict(color=T.s3, width=3), yaxis="y2",
                hovertemplate="%{x}<br>%{y:,} active<extra></extra>")

viz.style(fig, T, barmode="relative", height=420,
          title=dict(text=f"Members joined and lost each month ({FLOW_DISPLAY_START} on), "
                          "with total active members"),
          yaxis=dict(title="members joined / churned per month"), xaxis=dict(title=""))
# Independent right-hand axis with headroom above the data, so the (much larger, always-growing)
# active-member line reads as its own layer stacked above the bars rather than something sharing
# the bars' 0-centred scale -- that's the "gap" that keeps the two readable together.
fig.update_layout(yaxis2=dict(title="total active members", overlaying="y", side="right",
                              showgrid=False, rangemode="tozero",
                              range=[0, flow_display.active_total.max() * 1.15]))
fig.add_hline(y=0, line_color=T.axis, line_width=1)
fig.show()
flow_display.T

,2020-02,2020-03,2020-04,2020-05,2020-06,2020-07,2020-08,2020-09,2020-10,2020-11,2020-12,2021-01,2021-02,2021-03,2021-04,2021-05,2021-06,2021-07,2021-08,2021-09,2021-10,2021-11,2021-12,2022-01,2022-02,2022-03,2022-04,2022-05,2022-06,2022-07,...,2024-02,2024-03,2024-04,2024-05,2024-06,2024-07,2024-08,2024-09,2024-10,2024-11,2024-12,2025-01,2025-02,2025-03,2025-04,2025-05,2025-06,2025-07,2025-08,2025-09,2025-10,2025-11,2025-12,2026-01,2026-02,2026-03,2026-04,2026-05,2026-06,2026-07
joined,157,176,111,82,124,173,115,94,89,106,100,86,83,160,290,277,251,197,178,197,129,106,81,68,80,92,124,126,97,85,...,312,251,223,175,146,121,141,147,224,167,132,145,417,664,475,462,371,309,278,360,278,202,142,220,345,649,466,365,321,380
churned,65,91,99,96,58,76,101,72,68,68,72,79,75,51,59,79,103,91,112,78,88,98,90,82,76,78,79,82,82,93,...,129,200,160,159,153,147,145,113,154,104,124,143,105,187,251,292,314,361,341,301,325,282,218,238,219,246,333,401,0,0
net,92,85,12,-14,66,97,14,22,21,38,28,7,8,109,231,198,148,106,66,119,41,8,-9,-14,4,14,45,44,15,-8,...,183,51,63,16,-7,-26,-4,34,70,63,8,2,312,477,224,170,57,-52,-63,59,-47,-80,-76,-18,126,403,133,-36,321,380
active_total,2472,2557,2569,2555,2621,2718,2732,2754,2775,2813,2841,2848,2856,2965,3196,3394,3542,3648,3714,3833,3874,3882,3873,3859,3863,3877,3922,3966,3981,3973,...,5518,5569,5632,5648,5641,5615,5611,5645,5715,5778,5786,5788,6100,6577,6801,6971,7028,6976,6913,6972,6925,6845,6769,6751,6877,7280,7413,7377,7698,8078


In [15]:
if len(flow_display):
    _peak = flow_display.joined.idxmax()
    _rest = flow_display.joined[flow_display.joined.index != _peak]
    _trough = _rest.idxmin() if len(_rest) else None
    _first_neg = flow_display[flow_display.net < 0].index.min() if (flow_display.net < 0).any() else None

    bullets = [f"**Reading — signups peaked in {_peak} ({int(flow_display.loc[_peak, 'joined'])} joins)**"
              + (f", the slowest month on record is {_trough} "
                 f"({int(flow_display.loc[_trough, 'joined'])} joins)."
                 if _trough is not None else ".")]
    bullets.append(
        (f"**So-what — the book first went net-negative in {_first_neg}** "
         f"({int(flow_display.loc[_first_neg, 'net'])} net change) — churn outpaced signups that month.")
        if _first_neg is not None else
        "**So-what — this book has never gone net-negative** in the observed window."
    )
    bullets.append(
        f"**Reading — total active members ended the window at {int(flow_display.active_total.iloc[-1]):,}**, "
        f"up from {int(flow_display.active_total.iloc[0]):,} at the start of {FLOW_DISPLAY_START}."
    )
    bullets.append(f"*Caveat:* {FLOW_DISPLAY_START} on is shown here -- the export's very first "
                   "month is excluded from the bars as a likely bulk-backfill artifact (see the "
                   "code comment above), though its members still count in the active-total line.")
else:
    bullets = ["**Reading — no completed signup cohorts to plot yet.**"]
insight(bullets)

**Insights**

- **Reading — signups peaked in 2023-03 (920 joins)**, the slowest month on record is 2022-08 (53 joins).
- **So-what — the book first went net-negative in 2020-05** (-14 net change) — churn outpaced signups that month.
- **Reading — total active members ended the window at 8,078**, up from 2,472 at the start of 2020-02.
- *Caveat:* 2020-02 on is shown here -- the export's very first month is excluded from the bars as a likely bulk-backfill artifact (see the code comment above), though its members still count in the active-total line.

In [16]:
mw = wsh.set_index("event_date").resample("MS").size().rename("washes")
mc = pay[~pay.is_refund].set_index("event_date").resample("MS").size().rename("paid cycles")
act = pd.concat([mw, mc], axis=1).fillna(0).astype(int)

fig = go.Figure()
fig.add_scatter(x=act.index, y=act["washes"], name="Washes", mode="lines",
                line=dict(color=T.s1, width=2), hovertemplate="%{x|%b %Y}<br>%{y} washes<extra></extra>")
fig.add_scatter(x=act.index, y=act["paid cycles"], name="Paid cycles", mode="lines",
                line=dict(color=T.s3, width=2), hovertemplate="%{x|%b %Y}<br>%{y} cycles<extra></extra>")
viz.style(fig, T, height=360, title=dict(text="Monthly volume: washes vs paid membership months"),
          yaxis=dict(title="count"), xaxis=dict(title=""))
fig.show()

summary = pd.Series({
    "members (ever)": len(cust),
    f"active at {asof.date()}": int(cust.active.sum()),
    "churned": int(cust.churned.sum()),
    "lifetime churn rate": f"{cust.churned.mean():.1%}",
    "net revenue": f"${pay.amount.sum():,.0f}",
    "active-book MRR": f"${cust[cust.active].arpu.sum():,.0f}",
    "washes delivered": len(wsh),
    "washes per paid cycle": round(len(wsh) / max((~pay.is_refund).sum(), 1), 2),
}, name="value").to_frame()
summary

,value
members (ever),20277
active at 2026-07-31,8078
churned,12199
lifetime churn rate,60.2%
net revenue,"$16,559,252"
active-book MRR,"$330,716"
washes delivered,1290716
washes per paid cycle,3.4


In [17]:
_wpc = len(wsh) / max((~pay.is_refund).sum(), 1)
_active_n = max(int(cust.active.sum()), 1)
_arpu_book = cust[cust.active].arpu.sum() / _active_n
insight([
    f"**Reading — washes per paid membership month averages {_wpc:.2f}** across the export.",
    f"**So-what — active-book MRR is ${cust[cust.active].arpu.sum():,.0f} from {_active_n} active "
    f"members** (~${_arpu_book:,.0f} average ARPU).",
    "*Caveat:* the most recent calendar month may still be partial — read the last point on both "
    "lines above as incomplete.",
])

**Insights**

- **Reading — washes per paid membership month averages 3.40** across the export.
- **So-what — active-book MRR is $330,716 from 8078 active members** (~$41 average ARPU).
- *Caveat:* the most recent calendar month may still be partial — read the last point on both lines above as incomplete.

---
## 4. Retention: cohorts and the renewal hazard

Two views of the same thing. The **cohort table** answers "how many of the people who joined in
month X are still here?"; the **hazard curve** answers "at which membership month do we lose them?".
The second is the actionable one.

Both respect censoring: a member who joined three weeks ago has not *survived* three weeks, they are
simply unobserved, and never count in a denominator. `renewal_panel()` also assumes a ~30-day
billing cycle (`CYCLE_DAYS`) and a 45-day renewal grace window (`RENEW_WINDOW`) — inherited from V1,
checked against this export's own charge-gap distribution right after the hazard curve below, not
assumed.

In [18]:
ret = P.cohort_retention(ev)
z = ret.values.astype(float)
n_cohorts = len(ret.index)
# V1's fixed height=420 assumed ~12 monthly cohorts (Hurricane's ~1-year book). A longer-running
# book has far more signup-month rows, and squeezing them into a fixed height shrinks each row
# below its xgap/ygap, which is what turns the cell borders into a striped/moire pattern and stacks
# every cell's % annotation on top of its neighbours. Scale height to the actual row count instead,
# and drop the per-cell text once there are too many rows for it to stay legible either way --
# the hover tooltip still carries the same number.
row_h = 20
fig_height = max(420, min(2200, row_h * n_cohorts + 100))
show_cell_labels = n_cohorts <= 30

fig = go.Figure(go.Heatmap(
    z=z, x=[f"M{c}" for c in ret.columns], y=ret.index,
    colorscale=[[i / (len(T.seq) - 1), c] for i, c in enumerate(T.seq)],
    zmin=0, zmax=1, xgap=1, ygap=1, hoverongaps=False,
    colorbar=dict(title="retained", tickformat=".0%", outlinewidth=0, tickfont=dict(color=T.muted)),
    hovertemplate="%{y} cohort, %{x}<br>%{z:.0%} retained<extra></extra>"))
if show_cell_labels:
    for i, coh in enumerate(ret.index):
        for j, c in enumerate(ret.columns):
            v = ret.iloc[i, j]
            if pd.notna(v):
                fig.add_annotation(x=j, y=i, text=f"{v:.0%}", showarrow=False,
                                   font=dict(size=9, color="#ffffff" if v > 0.55 else T.ink))
viz.style(fig, T, height=fig_height, title=dict(text="Cohort retention by months since signup"
          + ("" if show_cell_labels else " (hover a cell for its %)")),
          xaxis=dict(title="months since signup", showgrid=False, type="category"),
          yaxis=dict(title="signup cohort", showgrid=False, type="category", autorange="reversed"))
fig.show()
print(f"{n_cohorts} signup cohorts plotted at {row_h}px/row (height={fig_height}px)"
      + ("" if show_cell_labels else "; per-cell labels hidden above 30 rows, use hover instead."))

79 signup cohorts plotted at 20px/row (height=1680px); per-cell labels hidden above 30 rows, use hover instead.


In [19]:
if 0 in ret.columns and 1 in ret.columns:
    _drop = (ret[0] - ret[1]).dropna()
    _avg_drop = _drop.mean() if len(_drop) else None
else:
    _avg_drop = None

insight([
    (f"**Reading — the average M0-to-M1 drop across cohorts is {_avg_drop:.0%}.**"
     if _avg_drop is not None else
     "**Reading — not enough cohorts have both M0 and M1 observed yet to characterise the early drop.**"),
    "*Caveat:* cells for young cohorts or late months are thin or unobserved by construction — read "
    "only cells backed by a reasonable member count as trend, same rule as V1.",
])

**Insights**

- **Reading — the average M0-to-M1 drop across cohorts is 9%.**
- *Caveat:* cells for young cohorts or late months are thin or unobserved by construction — read only cells backed by a reasonable member count as trend, same rule as V1.

In [20]:
panel = P.renewal_panel(ev, churn_after=CHURN_AFTER_V2)
haz_full = P.hazard_curve(panel)
o = panel[~panel.censored]

# Past ~month 90-100 this book only has a handful of members at that tenure (an 8-year-old book
# with monthly cycles), so each point there is n=5-10 and one dropout swings the rate 15-20 points
# -- noise, not signal (see the dip-and-recover pattern before this cap was added). Capping to the
# first MAX_MONTH_SHOWN months keeps the chart to the region with enough n per point to trust.
MAX_MONTH_SHOWN = 50
haz = haz_full[haz_full.index <= MAX_MONTH_SHOWN]
print(f"Showing membership months 0-{MAX_MONTH_SHOWN} ({len(haz)} of {len(haz_full)} points with "
      f"n>=5); the full curve runs out to month {haz_full.index.max() if len(haz_full) else 0}.")

# V1's fixed dtick=1 + one "n=..." annotation per point assumed ~10-11 membership months
# (Hurricane's ~1-year book). Thin both to the actual point count instead; the hover tooltip still
# carries every point's n regardless.
n_points = len(haz)
show_n_labels = n_points <= 30
x_dtick = max(1, round(n_points / 25)) if n_points else 1

fig = go.Figure()
if n_points:
    # n plotted directly as background bars on a right-hand axis, rather than only available on
    # hover -- gets its own generous headroom (3x the actual max) so the bars stay low and don't
    # compete visually with the renewal-rate line, which is what we still care about most.
    fig.add_bar(x=haz.index, y=haz.n, name="n (sample size)", marker_color=T.grid,
                marker_line=dict(width=0), opacity=0.7, yaxis="y2",
                hovertemplate="month %{x}<br>n=%{y}<extra></extra>")
    fig.add_scatter(x=haz.index, y=haz.renewal_rate, mode="lines+markers", name="Renewal rate",
                    line=dict(color=T.s1, width=2), marker=dict(size=9 if show_n_labels else 5),
                    customdata=haz.n, yaxis="y1",
                    hovertemplate="month %{x}<br>%{y:.1%} renew (n=%{customdata})<extra></extra>")
    if len(o):
        fig.add_hline(y=o.renewed.mean(), line_dash="dot", line_color=T.muted,
                      annotation_text=f"book average {o.renewed.mean():.1%}",
                      annotation_font=dict(color=T.muted, size=11))
    if show_n_labels:
        for x, r, n in zip(haz.index, haz.renewal_rate, haz.n):
            fig.add_annotation(x=x, y=r, text=f"n={n}", yshift=-18, showarrow=False,
                               font=dict(size=9, color=T.muted))
viz.style(fig, T, height=420, showlegend=True,
          title=dict(text=f"Renewal rate by membership month, 0-{MAX_MONTH_SHOWN} (censored cycles excluded)"
          + ("" if show_n_labels else " -- hover a point for its n")),
          yaxis=dict(title="renewed next cycle", tickformat=".0%"),
          xaxis=dict(title="membership month", dtick=x_dtick, range=[-1, MAX_MONTH_SHOWN + 1]))
fig.update_layout(yaxis2=dict(title="n (sample size)", overlaying="y", side="right",
                              showgrid=False, rangemode="tozero",
                              range=[0, haz.n.max() * 3 if n_points else 1]))
fig.show()
print(f"{n_points} membership-month points plotted"
      + ("" if show_n_labels else "; per-point n= labels hidden above 30 points, use hover instead."))
haz.round(3)

Showing membership months 0-50 (51 of 116 points with n>=5); the full curve runs out to month 115.


51 membership-month points plotted; per-point n= labels hidden above 30 points, use hover instead.


,n,renewal_rate
cycle_no,,
0,19144,0.891
1,17247,0.905
2,15709,0.906
3,14305,0.922
4,13162,0.929
5,12295,0.936
6,11624,0.943
7,11098,0.942
8,10539,0.948


In [21]:
bullets = []
if 0 in haz.index and (o.cycle_no > 0).any():
    _m0 = haz.loc[0, "renewal_rate"]
    _later = o[o.cycle_no > 0].renewed.mean()
    try:
        _chi = stats.chi2_contingency(pd.crosstab(o.cycle_no == 0, o.renewed))
        _sig = "significant" if _chi[1] < 0.05 else "not significant at the usual 0.05 threshold"
        _p_str = f"p={_chi[1]:.2g}"
    except ValueError:
        _sig, _p_str = "untestable (degenerate table)", "p=n/a"
    bullets.append(
        f"**Reading — the signup month (month 0) renews at {_m0:.1%} against {_later:.1%} for every "
        f"later month** — a {abs(_later - _m0) * 100:.1f}-point gap, chi2 {_p_str} ({_sig})."
    )
else:
    bullets.append("**Reading — not enough resolved month-0 cycles yet to compare against later months.**")
bullets.append(f"*Caveat:* {int(panel.censored.sum())} of {len(panel)} cycles ({panel.censored.mean():.1%}) "
               "are censored and excluded from every rate above.")
insight(bullets)

**Insights**

- **Reading — the signup month (month 0) renews at 89.1% against 95.5% for every later month** — a 6.5-point gap, chi2 p=0 (significant).
- *Caveat:* 8147 of 379199 cycles (2.1%) are censored and excluded from every rate above.

In [22]:
# Cycle-length sanity check: V1's CYCLE_DAYS=30 / RENEW_WINDOW=45 were fit to the Hurricane export,
# not derived from first principles. Check the actual gap between a customer's consecutive charges
# on THIS export before trusting those constants downstream.
_gaps = (pay[~pay.is_refund].sort_values(["customer_id", "event_date"])
         .groupby("customer_id")["event_date"].diff().dt.days.dropna())
if len(_gaps):
    _median_gap, _p75_gap = _gaps.median(), _gaps.quantile(0.75)
    _close_enough = abs(_median_gap - P.CYCLE_DAYS) <= 5
    insight([
        f"**Reading — the observed median gap between consecutive charges is {_median_gap:.0f} days "
        f"(p75 {_p75_gap:.0f} days)**, against the assumed `CYCLE_DAYS={P.CYCLE_DAYS}`.",
        (f"**So-what — {P.CYCLE_DAYS}-day cycles look like a reasonable fit for this export too.**"
         if _close_enough else
         f"**So-what — this export's billing cycle looks different from {P.CYCLE_DAYS} days.** "
         "`CYCLE_DAYS` and `RENEW_WINDOW` in `profiling.py` are shared module constants fit to the "
         "Hurricane export and still at their V1 defaults here (unlike `CHURN_AFTER`, already "
         "overridden to 90 above); if the gap is far from 30, treat renewal/churn numbers as "
         "provisional until those two are revisited as well."),
    ])
else:
    insight(["**Reading — fewer than two charges per customer on average; no renewal cadence to check yet.**"])

**Insights**

- **Reading — the observed median gap between consecutive charges is 30 days (p75 31 days)**, against the assumed `CYCLE_DAYS=30`.
- **So-what — 30-day cycles look like a reasonable fit for this export too.**

---
## 5. What drives churn

The same five hypotheses V1 tested, each run fresh on this export's own renewal panel — one row per
paid membership month, features measured at the moment of the charge, label is whether the *next*
charge arrived. Censored cycles are excluded from every rate.

In [23]:
def rate_by(frame, bucket, label):
    g = frame.groupby(bucket, observed=True).agg(n=("renewed", "size"), renewal=("renewed", "mean"))
    return g.rename_axis(label)


def chi2_safe(a, b):
    try:
        return stats.chi2_contingency(pd.crosstab(a, b))[1]
    except ValueError:
        return float("nan")


b = pd.cut(o.washes_this_cycle, [-1, 0, 1, 2, 4, 8, 1000], labels=["0", "1", "2", "3-4", "5-8", "9+"])
dorm = rate_by(o, b, "washes in the cycle just paid for")

fig = go.Figure(go.Bar(
    x=dorm.index.astype(str), y=dorm.renewal, marker_color=T.s1,
    text=[f"{v:.1%}" for v in dorm.renewal], textposition="outside", textfont=dict(color=T.ink2),
    customdata=dorm.n, marker_line=dict(width=2, color=T.surface),
    hovertemplate="%{x} washes<br>%{y:.1%} renew (n=%{customdata})<extra></extra>"))
viz.style(fig, T, height=380, showlegend=False,
          title=dict(text="a) Dormancy — do members who didn't wash renew less?"),
          yaxis=dict(title="renewal rate", tickformat=".0%"),
          xaxis=dict(title="washes in the 30 days ending at the charge"))
fig.show()

_p = chi2_safe(o.dormant, o.renewed)
_dorm_rate = o[o.dormant].renewed.mean() if o.dormant.any() else float("nan")
_active_rate = o[~o.dormant].renewed.mean() if (~o.dormant).any() else float("nan")
insight([
    (f"**Reading — zero-wash cycles renew at {_dorm_rate:.1%} vs {_active_rate:.1%} for any-wash "
     f"cycles** (chi2 p={_p:.2g}, {'significant' if _p < 0.05 else 'not significant'})."
     if pd.notna(_p) else "**Reading — not enough variation to test dormancy vs renewal yet.**"),
    f"**So-what — dormant cycles are {o.dormant.mean():.1%} of all paid cycles**, worth "
    f"${o[o.dormant].amount.sum():,.0f} ({o[o.dormant].amount.sum() / max(o.amount.sum(), 1):.1%} "
    "of cycle revenue).",
])
dorm.round(3)

**Insights**

- **Reading — zero-wash cycles renew at 90.0% vs 96.1% for any-wash cycles** (chi2 p=0, significant).
- **So-what — dormant cycles are 14.9% of all paid cycles**, worth $1,987,539 (12.1% of cycle revenue).

,n,renewal
washes in the cycle just paid for,,
0,55432,0.900
1,55949,0.945
2,64426,0.954
3-4,92738,0.963
5-8,71939,0.970
9+,30568,0.973


In [24]:
oo = o[o.prev_amount.notna()]
if len(oo) and oo.price_step_up.any() and (~oo.price_step_up).any():
    step = pd.DataFrame({
        "renewal": [oo[oo.price_step_up].renewed.mean(), oo[~oo.price_step_up].renewed.mean()],
        "n": [int(oo.price_step_up.sum()), int((~oo.price_step_up).sum())],
    }, index=["price stepped up >15%", "price held flat"])
    _p2 = chi2_safe(oo.price_step_up, oo.renewed)

    fig = go.Figure(go.Bar(
        x=step.index, y=step.renewal, marker_color=[T.s2, T.s1],
        text=[f"{v:.1%}" for v in step.renewal], textposition="outside", textfont=dict(color=T.ink2),
        customdata=step.n, marker_line=dict(width=2, color=T.surface),
        hovertemplate="%{x}<br>%{y:.1%} renew (n=%{customdata})<extra></extra>"))
    viz.style(fig, T, height=340, showlegend=False,
              title=dict(text="b) The promo cliff — does a price jump cost renewal?"),
              yaxis=dict(title="renewal rate", tickformat=".0%"), xaxis=dict(title=""))
    fig.show()

    insight([
        (f"**Reading — step-up cycles renew at {step.renewal.iloc[0]:.1%} vs "
         f"{step.renewal.iloc[1]:.1%} for flat charges** (chi2 p={_p2:.2g}, "
         f"{'significant' if _p2 < 0.05 else 'not significant'})."),
        f"**So-what — the median jump is ${oo[oo.price_step_up].prev_amount.median():.0f} -> "
        f"${oo[oo.price_step_up].amount.median():.0f}** among stepped-up charges.",
    ])
    step.round(3)
else:
    insight(["**Reading — not enough stepped-up charges in this export to test the promo cliff.**"])

**Insights**

- **Reading — step-up cycles renew at 90.7% vs 95.8% for flat charges** (chi2 p=2e-274, significant).
- **So-what — the median jump is $23 -> $40** among stepped-up charges.

In [25]:
onv = o[~o.no_vehicle_on_file]
if len(onv) and (onv.n_vehicles_billed > 1).any() and (onv.n_vehicles_billed == 1).any():
    veh = rate_by(onv, onv.n_vehicles_billed.clip(upper=4), "vehicles on the account")
    veh.index = [str(i) for i in sorted(veh.index.astype(int))] if veh.index.dtype != object else veh.index
    _p3 = chi2_safe(onv.n_vehicles_billed > 1, onv.renewed)

    fig = go.Figure(go.Bar(
        x=veh.index.astype(str), y=veh.renewal, marker_color=T.s1,
        text=[f"{v:.1%}" for v in veh.renewal], textposition="outside", textfont=dict(color=T.ink2),
        customdata=veh.n, marker_line=dict(width=2, color=T.surface),
        hovertemplate="%{x} vehicles<br>%{y:.1%} renew (n=%{customdata})<extra></extra>"))
    viz.style(fig, T, height=340, showlegend=False,
              title=dict(text="c) Household size — does a second vehicle change renewal?"),
              yaxis=dict(title="renewal rate", tickformat=".0%"), xaxis=dict(title="vehicles billed on the account"))
    fig.show()

    _one = onv[onv.n_vehicles_billed == 1].renewed.mean()
    _multi = onv[onv.n_vehicles_billed > 1].renewed.mean()
    insight([
        (f"**Reading — 1 vehicle renews at {_one:.1%} vs {_multi:.1%} for 2+ vehicles** "
         f"(chi2 p={_p3:.2g}, {'significant' if _p3 < 0.05 else 'not significant'})."),
    ])
    veh.round(3)
else:
    insight(["**Reading — not enough multi-vehicle households in this export to test household size.**"])

**Insights**

- **Reading — 1 vehicle renews at 93.4% vs 97.6% for 2+ vehicles** (chi2 p=0, significant).

In [26]:
if o.joined_on_promo.any() and (~o.joined_on_promo).any():
    _p4 = chi2_safe(o.joined_on_promo, o.renewed)
    promo = pd.DataFrame({
        "renewal": [o[o.joined_on_promo].renewed.mean(), o[~o.joined_on_promo].renewed.mean()],
        "n": [int(o.joined_on_promo.sum()), int((~o.joined_on_promo).sum())],
    }, index=["joined on promo", "joined at full price"])
    print(promo.round(3).to_string())
    insight([
        (f"**Reading — promo joiners renew at {promo.renewal.iloc[0]:.1%}, full-price joiners at "
         f"{promo.renewal.iloc[1]:.1%}** (chi2 p={_p4:.2g}, "
         f"{'a real difference' if _p4 < 0.05 else 'not significant -- read as a null result'})."),
        f"**Caveat:** promo share of signups is {cust.joined_on_promo.mean():.1%} on this export -- "
        "if that's very high or very low, the full-price comparison group may be too small to trust.",
    ])
else:
    insight(["**Reading — every signup in this export is on the same pricing track (all-promo or "
             "all-full-price), so promo-vs-full-price cannot be tested here.**"])

                      renewal       n
joined on promo         0.924   82497
joined at full price    0.960  288555


**Insights**

- **Reading — promo joiners renew at 92.4%, full-price joiners at 96.0%** (chi2 p=0, a real difference).
- **Caveat:** promo share of signups is 39.2% on this export -- if that's very high or very low, the full-price comparison group may be too small to trust.

In [27]:
pkg = (o.groupby("membership_package_name")
        .agg(n=("renewed", "size"), renewal=("renewed", "mean"),
             list_price=("current_package_price", "first"),
             washes=("washes_this_cycle", "mean"))
        .sort_values("n", ascending=False))
pkg.round(3)

,n,renewal,list_price,washes
membership_package_name,,,,
Ceramic Wash Monthly Membership Plan,118939,0.947,50.00,3.766
Ultimate Wash Monthly Membership Plan,83897,0.959,40.00,3.466
Better Wash Monthly Membership Plan,71570,0.953,35.00,3.398
Good Wash Monthly Membership Plan,70841,0.946,30.00,3.643
Ceramic Military | First Responder Monthly Membership Plan,24509,0.973,38.00,4.062
Ceramic 1 Monthly Unlimited,477,0.922,50.00,4.025
JJ CERAMIC MARKETING PLAN,221,0.493,0.01,4.606
Cline Annual Plan,119,0.521,0.01,2.252
Windsong Annual Plan Comp,96,0.823,NaN,6.750


In [28]:
if len(pkg) > 1:
    _best = pkg.renewal.idxmax()
    _worst = pkg.renewal.idxmin()
    insight([
        f"**Reading — renewal ranges from {pkg.renewal.min():.1%} ({_worst}) to "
        f"{pkg.renewal.max():.1%} ({_best})** across {len(pkg)} packages with volume in this export.",
        "*Caveat:* packages with a handful of cycles (`n` in the single digits) are anecdotes, not "
        "segments — read the `n` column before trusting any row here.",
    ])
else:
    insight(["**Reading — this export only has one membership package with paid-cycle volume, so "
             "package tier cannot be compared.**"])

**Insights**

- **Reading — renewal ranges from 0.0% (Ultimate Annual Plan (12 Mos for Price of 11Mos)) to 100.0% (JJ's Ceramic Marketing Annual 3 Member)** across 26 packages with volume in this export.
- *Caveat:* packages with a handful of cycles (`n` in the single digits) are anecdotes, not segments — read the `n` column before trusting any row here.

---
## 6. A CM

**Frame:** discrete-time renewal hazard, same as V1 — one row per paid membership month, label is
whether the next charge arrived within `RENEW_WINDOW` days, features are only what was knowable at
the moment of the charge.

**Why logistic regression:** on a panel this size, a linear model's coefficients are half the
deliverable — *which lever*, not just a ranking. A LightGBM check right after confirms whether a
booster is finding anything the linear model misses; if the panel here turns out too small or too
imbalanced to fit reliably, both cells say so rather than raising an opaque error.

In [29]:
try:
    mod = P.fit_churn_model(panel)
    print(f"train {mod.n_train} cycles  ->  holdout {mod.n_test} cycles after {mod.cutoff.date()}")
    print(f"AUC   5-fold CV {mod.auc_cv:.3f}   |   time-ordered holdout {mod.auc_holdout:.3f}")
    print(f"base churn per cycle {mod.base_rate:.1%}   |   top-decile lift {mod.top_decile_lift:.2f}x")
    _model_ok = True
except Exception as exc:  # noqa: BLE001 -- an unseen export may be too small/imbalanced to fit
    print(f"Could not fit a churn model on this export's panel: {exc!r}")
    print("This usually means too few resolved cycles or too few churn events for a time-ordered "
          "holdout split -- worth revisiting once more history has accumulated.")
    mod = None
    _model_ok = False

if _model_ok:
    try:
        import lightgbm as lgb
        from sklearn.metrics import roc_auc_score
        from sklearn.model_selection import StratifiedKFold

        d = P._model_frame(panel[~panel.censored])
        X, y = d[P.FEATURES], d.renewed.astype(int)
        aucs = []
        for tr, te in StratifiedKFold(5, shuffle=True, random_state=0).split(X, y):
            g = lgb.LGBMClassifier(n_estimators=200, learning_rate=0.05, num_leaves=7,
                                   min_child_samples=40, verbose=-1)
            g.fit(X.iloc[tr], y.iloc[tr])
            aucs.append(roc_auc_score(y.iloc[te], g.predict_proba(X.iloc[te])[:, 1]))
        print(f"LightGBM 5-fold CV AUC {np.mean(aucs):.3f}  (vs logistic {mod.auc_cv:.3f})")
    except Exception as exc:  # noqa: BLE001
        print(f"LightGBM comparison skipped: {exc!r}")

train 278289 cycles  ->  holdout 92763 cycles after 2025-05-29
AUC   5-fold CV 0.709   |   time-ordered holdout 0.735
base churn per cycle 4.8%   |   top-decile lift 3.49x
LightGBM 5-fold CV AUC 0.724  (vs logistic 0.709)


In [30]:
if _model_ok:
    insight([
        f"**Reading — holdout AUC is {mod.auc_holdout:.3f}** on a time-ordered split (train up to "
        f"{mod.cutoff.date()}, predict the {mod.n_test} cycles after).",
        f"**So-what — top-decile lift is {mod.top_decile_lift:.2f}x** the base churn rate of "
        f"{mod.base_rate:.1%} — the number that makes a targeted save campaign cheaper than a "
        "blanket one, if it holds up.",
        ("*Caveat:* AUC is useful for triage, not individual verdicts, regardless of how high it "
         "reads here — most of the signal in V1 was recency and price-step; this export may have "
         "a different mix or none of the same drivers."),
    ])
else:
    insight(["**No model to read yet** — see the message above for why fitting didn't complete."])

**Insights**

- **Reading — holdout AUC is 0.735** on a time-ordered split (train up to 2025-05-29, predict the 92763 cycles after).
- **So-what — top-decile lift is 3.49x** the base churn rate of 4.8% — the number that makes a targeted save campaign cheaper than a blanket one, if it holds up.
- *Caveat:* AUC is useful for triage, not individual verdicts, regardless of how high it reads here — most of the signal in V1 was recency and price-step; this export may have a different mix or none of the same drivers.

In [31]:
if _model_ok:
    orat = mod.odds_ratios()
    colors = [T.critical if v > 1 else T.good for v in orat]
    fig = go.Figure(go.Bar(
        x=orat.values - 1, y=orat.index, orientation="h", marker_color=colors,
        marker_line=dict(width=2, color=T.surface),
        text=[f"{v:.2f}x" for v in orat], textposition="outside", textfont=dict(color=T.ink2),
        hovertemplate="%{y}<br>%{text} churn odds per +1 SD<extra></extra>"))
    viz.style(fig, T, height=420, showlegend=False,
              title=dict(text="Churn odds multiplier per +1 SD  (right = raises churn)"),
              xaxis=dict(title="odds multiplier"), yaxis=dict(title="", autorange="reversed"))
    fig.add_vline(x=0, line_color=T.axis, line_width=1)
    fig.show()

    _top = orat.index[0]
    _bottom = orat.index[-1]
    insight([
        f"**Reading — `{_top}` has the strongest churn-raising effect** ({orat.iloc[0]:.2f}x per SD); "
        f"`{_bottom}` has the strongest protective effect ({orat.iloc[-1]:.2f}x per SD).",
        "*Caveat:* `month` (if present among the features) absorbs calendar time on a young book and "
        "should not be read as a seasonality finding without more history.",
    ])
    orat.round(2).to_frame("churn odds per +1 SD")

**Insights**

- **Reading — `days_since_wash` has the strongest churn-raising effect** (1.26x per SD); `n_vehicles_billed` has the strongest protective effect (0.68x per SD).
- *Caveat:* `month` (if present among the features) absorbs calendar time on a young book and should not be read as a seasonality finding without more history.

In [32]:
if _model_ok:
    d = P._model_frame(panel[~panel.censored])
    risk = mod.score(d)
    try:
        q5 = pd.qcut(risk, 5, labels=["Q1 safest", "Q2", "Q3", "Q4", "Q5 riskiest"], duplicates="drop")
        cal = (pd.DataFrame({"pred": risk, "actual": 1 - d.renewed.astype(int).values})
               .groupby(q5, observed=True).agg(n=("actual", "size"), predicted=("pred", "mean"),
                                              actual=("actual", "mean")))
        fig = go.Figure()
        fig.add_bar(x=cal.index.astype(str), y=cal.predicted, name="Predicted churn", marker_color=T.s1,
                    marker_line=dict(width=2, color=T.surface),
                    hovertemplate="%{x}<br>predicted %{y:.1%}<extra></extra>")
        fig.add_bar(x=cal.index.astype(str), y=cal.actual, name="Actual churn", marker_color=T.s2,
                    marker_line=dict(width=2, color=T.surface),
                    hovertemplate="%{x}<br>actual %{y:.1%}<extra></extra>")
        viz.style(fig, T, height=360, barmode="group",
                  title=dict(text="Calibration -- predicted vs realised churn by risk quintile"),
                  yaxis=dict(title="churn rate", tickformat=".0%"), xaxis=dict(title=""))
        fig.show()

        _gap = (cal.predicted - cal.actual).abs().max()
        insight([
            f"**Reading — predicted vs actual churn tracks within {_gap:.1%} across quintiles** "
            f"(riskiest: predicts {cal.predicted.iloc[-1]:.1%}, delivers {cal.actual.iloc[-1]:.1%}).",
            "*Caveat:* these are in-sample fitted values -- the honest out-of-sample evidence is the "
            "holdout AUC above, not this chart.",
        ])
        cal.round(3)
    except ValueError as exc:
        print(f"Could not build risk quintiles: {exc!r} -- likely too few resolved cycles.")

**Insights**

- **Reading — predicted vs actual churn tracks within 0.5% across quintiles** (riskiest: predicts 10.8%, delivers 11.1%).
- *Caveat:* these are in-sample fitted values -- the honest out-of-sample evidence is the holdout AUC above, not this chart.

---
## 7. Segmentation — four personas

K-means on six standardised behaviour-and-economics features (`washes_per_month`, `tenure_months`,
`arpu`, `n_vehicles`, `cost_per_wash`, `days_since_wash`), same as V1. Money features are
log-transformed; personas are matched to fixed archetypes by optimal assignment (see
`profiling.ARCHETYPES`), so a re-fit cannot silently swap two labels — but on a new export, whether
four *meaningfully separated* clusters even exist is itself something to check, not assume.

****Further Steps to add**** : Do one more thing before performing the acutal clustering do this : Take the features vector per customer, apply the t-SNE or UMAP to reduce it to 2D and showcase how does it looks like.

In [33]:
try:
    from sklearn.metrics import silhouette_score
    from sklearn.preprocessing import StandardScaler

    sil = {}
    for k in range(2, min(7, max(cust.cycles_paid.notna().sum(), 3))):
        dd, _ = P.segment(cust, k=k)
        Xs = dd[P.SEG_FEATURES].copy()
        Xs["days_since_wash"] = Xs.days_since_wash.fillna(120).clip(upper=120)
        for c in ["arpu", "cost_per_wash", "washes_per_month"]:
            Xs[c] = np.log1p(Xs[c].clip(lower=0))
        sil[k] = silhouette_score(StandardScaler().fit_transform(Xs), dd.segment_id)
    print("silhouette by k:", {k: round(v, 3) for k, v in sil.items()})

    seg, prof = P.segment(cust, k=4)
    order = [s for s in viz.SEGMENT_ORDER if s in prof.index]
    _seg_ok = True
except Exception as exc:  # noqa: BLE001 -- too few customers with economics to cluster
    print(f"Could not segment this export: {exc!r}")
    print("This usually means too few customers with a payment history (cycles_paid notna) to "
          "support k=4 clusters.")
    seg, prof, order = None, None, []
    _seg_ok = False

if _seg_ok:
    prof[["members", "washes_per_month", "tenure_months", "arpu", "n_vehicles",
          "cost_per_wash", "days_since_wash", "churn_rate", "revenue_share"]].round(2)

silhouette by k: {2: 0.306, 3: 0.243, 4: 0.284, 5: 0.294, 6: 0.295}


In [34]:
if _seg_ok:
    _best_k = max(sil, key=sil.get)
    _highest_churn = prof.churn_rate.idxmax()
    _biggest_rev_share = prof.revenue_share.idxmax()
    _member_share = prof.loc[_biggest_rev_share, "members"] / prof.members.sum()
    insight([
        f"**Reading — k=4 scores silhouette {sil.get(4, float('nan')):.3f}** "
        f"(best of the range tried is k={_best_k} at {sil[_best_k]:.3f}) -- a judgement call, same "
        "as V1, not a discovered truth.",
        f"**Reading — `{_highest_churn}` has the highest churn rate** in this segmentation "
        f"({prof.loc[_highest_churn, 'churn_rate']:.0%}).",
        f"**So-what — `{_biggest_rev_share}` is {_member_share:.0%} of members but "
        f"{prof.loc[_biggest_rev_share, 'revenue_share']:.0%} of revenue** -- the concentration V1 "
        "found may or may not repeat here; the gap between those two percentages is the thing to "
        "watch.",
    ])
else:
    insight(["**No personas to read yet** — see the message above."])

**Insights**

- **Reading — k=4 scores silhouette 0.284** (best of the range tried is k=2 at 0.306) -- a judgement call, same as V1, not a discovered truth.
- **Reading — `Promo flipper` has the highest churn rate** in this segmentation (98%).
- **So-what — `Power household` is 13% of members but 49% of revenue** -- the concentration V1 found may or may not repeat here; the gap between those two percentages is the thing to watch.

**Response — 2D projection of the clustering features, coloured by the fitted k-means clusters**

Same 6 standardized features `P.segment()` clustered on (`P.SEG_FEATURES`, log-transformed
money/frequency terms, days-since-wash filled/clipped), reduced to 2D with t-SNE — moved to run
**after** clustering (per follow-up) and coloured by the `segment` label k-means actually assigned,
using the same house colours as every other persona chart in this notebook. This is the validation
view: do the 4 personas visually separate in 2D, or overlap enough that "4 clusters" is a less
clean story than the silhouette score above suggests?

Used t-SNE rather than UMAP: t-SNE ships with scikit-learn, already a dependency here, while UMAP
would need a new package installed (`umap-learn`) — out of scope for a notebook-only pass.

*Caveat, since it's easy to over-read a t-SNE plot:* distances **within** a colour are meaningful
(nearby points are genuinely similar on these 6 features), but distances **between** colours and
each blob's size/shape are not reliable — t-SNE preserves local neighborhoods, not global geometry,
and a different random seed can rearrange the whole layout. Clean separation here is a good sign;
overlap doesn't necessarily mean the clusters are wrong, just not perfectly separable on 2
arbitrary projected axes.

In [35]:
if _seg_ok:
    from sklearn.manifold import TSNE
    from sklearn.preprocessing import StandardScaler

    _d = seg.copy()   # already carries "segment"/"segment_id" from P.segment() above
    _X = _d[P.SEG_FEATURES].copy()
    _X["days_since_wash"] = _X.days_since_wash.fillna(120).clip(upper=120)
    for _c in ["arpu", "cost_per_wash", "washes_per_month"]:
        _X[_c] = np.log1p(_X[_c].clip(lower=0))
    _Z = StandardScaler().fit_transform(_X)

    try:
        _perplexity = min(30, max(5, len(_d) // 20))
        _emb = TSNE(n_components=2, random_state=0, perplexity=_perplexity, init="pca").fit_transform(_Z)

        fig = go.Figure()
        for s in order:
            _mask = (_d.segment == s).values
            _sub, _sub_emb = _d[_mask], _emb[_mask]
            fig.add_scatter(
                x=_sub_emb[:, 0], y=_sub_emb[:, 1], mode="markers", name=s,
                marker=dict(size=5, opacity=0.65, line=dict(width=0), color=SEG_COLOR.get(s, T.s1)),
                customdata=np.stack([_sub.customer_id, _sub.arpu, _sub.washes_per_month,
                                     _sub.tenure_months], axis=-1),
                hovertemplate=(f"{s} · customer " + "%{customdata[0]}<br>ARPU $%{customdata[1]:.0f}"
                              "<br>%{customdata[2]:.1f} washes/mo<br>%{customdata[3]:.1f} mo tenure"
                              "<extra></extra>"))
        viz.style(fig, T, height=480, showlegend=True,
                  title=dict(text="t-SNE projection of the clustering features, coloured by persona "
                                  f"(n={len(_d)}, perplexity={_perplexity})"),
                  xaxis=dict(title="t-SNE 1", showgrid=False, zeroline=False),
                  yaxis=dict(title="t-SNE 2", showgrid=False, zeroline=False))
        fig.show()
    except Exception as exc:  # noqa: BLE001 -- t-SNE can be slow/unstable on very small or huge n
        print(f"Could not compute a t-SNE embedding: {exc!r}")
else:
    print("No segments to visualize -- see the clustering cell above.")

In [36]:
raw[raw["customer_id"] == 2863].sort_values(by="event_date")

,site_id,membership_package_name,customer_id,customer_no,vehicle_id,vehicle_license,vehicle_state,vehicle_make,vehicle_model,vehicle_year,vehicle_color,vehicle_vin,vehicle_type,vehicle_active,vehicle_black_listed,event_date,event_type,payment_type,amount,current_package_price
401211,3,Ceramic Wash Monthly Membership Plan,2863,063242031,NaN,NaN,NaN,NaN,NaN,NaN,None,None,NaN,NaN,NaN,2020-01-02 14:42:03,wash,NaN,NaN,NaN
401212,3,Ceramic Wash Monthly Membership Plan,2863,063242031,2375.0,DH7W023,NaN,GMC,YUKON XL,2009.0,None,None,NaN,1.0,0.0,2020-01-15 06:20:10,payment,renewal,44.0,50.0
401213,3,Ceramic Wash Monthly Membership Plan,2863,063242031,NaN,NaN,NaN,NaN,NaN,NaN,None,None,NaN,NaN,NaN,2020-01-25 18:59:36,wash,NaN,NaN,NaN
401214,3,Ceramic Wash Monthly Membership Plan,2863,063242031,NaN,NaN,NaN,NaN,NaN,NaN,None,None,NaN,NaN,NaN,2020-01-31 19:24:34,wash,NaN,NaN,NaN
401215,3,Ceramic Wash Monthly Membership Plan,2863,063242031,NaN,NaN,NaN,NaN,NaN,NaN,None,None,NaN,NaN,NaN,2020-02-14 20:01:54,wash,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
401301,3,Ceramic Wash Monthly Membership Plan,2863,063242031,NaN,NaN,NaN,NaN,NaN,NaN,None,None,NaN,NaN,NaN,2021-05-09 15:34:56,wash,NaN,NaN,NaN
401302,3,Ceramic Wash Monthly Membership Plan,2863,063242031,NaN,NaN,NaN,NaN,NaN,NaN,None,None,NaN,NaN,NaN,2021-05-15 14:50:46,wash,NaN,NaN,NaN
401303,3,Ceramic Wash Monthly Membership Plan,2863,063242031,NaN,NaN,NaN,NaN,NaN,NaN,None,None,NaN,NaN,NaN,2021-05-25 19:12:58,wash,NaN,NaN,NaN
401304,3,Ceramic Wash Monthly Membership Plan,2863,063242031,2375.0,DH7W023,NaN,GMC,YUKON XL,2009.0,None,None,NaN,1.0,0.0,2021-05-26 05:11:23,payment,renewal,45.0,50.0


In [37]:
cust[cust["customer_id"] == 21411]

,customer_id,site_id,package,list_price,n_vehicles,vehicle_type,vehicle_make,state,joined,last_payment,cycles_paid,revenue,refunds,signup_amount,last_amount,joined_on_promo,washes,last_wash,days_since_wash,days_since_payment,tenure_days,tenure_months,washes_per_month,washes_per_vehicle_month,arpu,cost_per_wash,active,churned,churn_month,cohort
13089,21411,4,Ceramic Wash Monthly Membership Plan,50.0,1,NaN,Ford,TX,2024-11-25 23:51:30,2025-01-25 10:09:06,3.0,150.0,0.0,50.0,50.0,False,4,2025-01-31 16:19:13,546.0,552.0,90.0,2.956636,1.352889,1.352889,50.0,37.5,False,True,2025-02,2024-11


In [38]:
if _seg_ok:
    # "name" (e.g. "Power household") is the persona label matched to an archetype; "segment_id"
    # is the raw k-means cluster number (0..k-1) behind it. Showing both on the axis makes it
    # traceable back to the actual cluster, not just the assigned name.
    order_labels = [f"{s} (C{int(prof.loc[s, 'segment_id'])})" for s in order]
    left_data = [prof.loc[s, "members"] / prof.members.sum() for s in order]
    right_data = [prof.loc[s, "revenue_share"] for s in order]
    fig = go.Figure()
    fig.add_bar(x=order_labels, y=left_data, name="Share of members", marker_color=T.s1,
                marker_line=dict(width=2, color=T.surface),
                hovertemplate="%{x}<br>%{y:.0%} of members<extra></extra>")
    fig.add_bar(x=order_labels, y=right_data, name="Share of revenue", marker_color=T.s2,
                marker_line=dict(width=2, color=T.surface),
                hovertemplate="%{x}<br>%{y:.0%} of revenue<extra></extra>")
    viz.style(fig, T, height=380, barmode="group",
              title=dict(text="Who they are vs what they are worth"),
              yaxis=dict(title="share", tickformat=".0%"), xaxis=dict(title=""))
    # Response to "add the car wash frequency for each cluster" -- annotated directly onto
    # this chart (not a separate table), one label per persona above its taller bar.
    for i, s in enumerate(order):
        _wpm = prof.loc[s, "washes_per_month"]
        fig.add_annotation(x=order_labels[i], y=max(left_data[i], right_data[i]), yshift=18,
                           showarrow=False, text=f"{_wpm:.1f} washes/mo",
                           font=dict(size=10, color=T.muted))
    fig.show()

In [39]:
# Average lifetime cust['revenue'] per cluster -- the plain historical-spend figure discussed in
# the CLV reasoning above, plotted directly rather than left as just a formula reference.
if _seg_ok:
    avg_rev = seg.groupby("segment")["revenue"].mean().reindex(order)
    order_labels_rev = [f"{s} (C{int(prof.loc[s, 'segment_id'])})" for s in order]
    fig = go.Figure(go.Bar(
        x=order_labels_rev, y=avg_rev.values, marker_color=[SEG_COLOR.get(s, T.s1) for s in order],
        marker_line=dict(width=2, color=T.surface),
        text=[f"${v:,.0f}" for v in avg_rev.values], textposition="outside", textfont=dict(color=T.ink2),
        hovertemplate="%{x}<br>avg lifetime revenue $%{y:,.0f}<extra></extra>"))
    viz.style(fig, T, height=380, showlegend=False,
              title=dict(text="Average lifetime revenue per cluster (cust['revenue'], mean)"),
              yaxis=dict(title="avg revenue ($)"), xaxis=dict(title=""))
    fig.show()
else:
    print("No segments to price -- see the clustering cell above.")

In [40]:
if _seg_ok:
    _hi, _lo = avg_rev.idxmax(), avg_rev.idxmin()
    _ratio = avg_rev.max() / max(avg_rev.min(), 1e-9)
    insight([
        f"**Reading — `{_hi}` has the highest average lifetime revenue** (${avg_rev.max():,.0f}), "
        f"`{_lo}` the lowest (${avg_rev.min():,.0f}) -- a {_ratio:.1f}x spread.",
        "**Reminder:** this is the plain historical-spend figure (`cust['revenue']`, mean per "
        "cluster) -- backward-looking, unlike the forward-looking CLV chart in §8 below. Compare "
        "the ranking here against that CLV ranking once you run it; agreement isn't guaranteed, "
        "since CLV also weighs wash cost and expected remaining lifetime, not just money already "
        "collected.",
    ])
else:
    insight(["**No segments to read yet.**"])

**Insights**

- **Reading — `Power household` has the highest average lifetime revenue** ($3,284), `Promo flipper` the lowest ($328) -- a 10.0x spread.
- **Reminder:** this is the plain historical-spend figure (`cust['revenue']`, mean per cluster) -- backward-looking, unlike the forward-looking CLV chart in §8 below. Compare the ranking here against that CLV ranking once you run it; agreement isn't guaranteed, since CLV also weighs wash cost and expected remaining lifetime, not just money already collected.

****Further Steps to add****
In this we need to add:
1. In the cell 32's output we further need to add the car wash frequency for each cluster.
2. Also the CLV should be total amount spend by the customers not the ARPU - (washes/month x variable cost per wash), I want to look at the reasoning of why this formula : (washes/month x variable cost per wash)


**Response — why CLV isn't "total amount spent"**

"Total amount spent" already exists in this notebook — it's `cust['revenue']` (and
`arpu = revenue / cycles_paid`), both used throughout §3 and this section's own economics table.
That number is **backward-looking**: what's already been collected from a customer. Useful for
describing the book as it stands, but it's a different question from what CLV is built to answer.

CLV is deliberately **forward-looking**: "starting today, how much more profit do we expect from a
typical member in this group before they eventually leave?" That's the number a decision like "how
much can we justify spending to retain this member" actually needs — money already banked doesn't
tell you that; expected future profit does.

Two things follow from being forward-looking, not historical:

1. **Why multiply by an expected lifetime, not just report a monthly number.** A member isn't worth
   one month's profit — they're worth however many months they're expected to keep paying.
   `expected_lifetime_months = 1 / monthly_churn` is the standard way to turn a monthly rate into an
   expected duration (5% monthly churn implies an average survivor sticks around ~20 more months).
2. **Why subtract wash cost instead of using raw ARPU.** The business doesn't keep 100% of what a
   member pays — every wash costs real money (water, chemicals, power, labour). A member paying
   $30/month but washing 20 times a month could be **costing** more in wash expense than they pay
   in — profitable-looking on revenue, unprofitable on margin. `contribution = arpu -
   (washes/month x cost/wash)` is what's actually left over after service cost, which is the number
   worth projecting forward, not the gross payment.

So: `revenue`/`arpu` (already here) answers "how much have we made from them so far." CLV answers
"how much more are we likely to make." Both are legitimate — collapsing them into one number would
lose the forward-looking, margin-aware property that makes CLV useful for a decision like a
retention-campaign ROI calculation.

*A "total historical spend per persona" figure alongside CLV (not instead of it) would be a small,
notebook-only addition if wanted. Changing the CLV **formula** itself lives in `profiling.py`'s
`unit_economics()`, which this pass doesn't touch (scoped to notebook-only edits) — flagging as a
decision for a follow-up rather than changing it unprompted.*

monthly_wash_cost       = washes_per_month × variable_cost_per_wash   # $4/wash by default

monthly_contribution    = arpu - monthly_wash_cost

expected_lifetime_months = 1 / monthly_churn

clv                      = monthly_contribution × expected_lifetime_months


In [41]:
cust

,customer_id,site_id,package,list_price,n_vehicles,vehicle_type,vehicle_make,state,joined,last_payment,cycles_paid,revenue,refunds,signup_amount,last_amount,joined_on_promo,washes,last_wash,days_since_wash,days_since_payment,tenure_days,tenure_months,washes_per_month,washes_per_vehicle_month,arpu,cost_per_wash,active,churned,churn_month,cohort
0,1,3,Ceramic Military | First Responder Monthly Mem...,38.00,10,Passenger Vehicle,Honda,TX,2020-01-09 06:05:00,2026-05-08 07:29:54,56.0,2621.06,-60.0,44.00,38.00,False,1086,2026-07-27 20:52:22,4.0,84.0,2341.0,76.905388,14.121247,1.412125,46.804643,2.413499,True,False,NaN,2020-01
1,6,1,JJ CERAMIC MARKETING PLAN,0.01,1,NaN,,TX,2023-03-25 18:35:32,2023-03-25 18:35:32,1.0,0.01,0.0,0.01,0.01,False,17,2024-03-22 19:59:16,861.0,1224.0,30.0,0.985545,17.000000,17.000000,0.010000,0.000588,False,True,2023-04,2023-03
2,8,1,Ceramic Wash Monthly Membership Plan,50.00,1,NaN,Ford,TX,2020-01-09 06:05:00,2025-04-28 10:29:10,66.0,2976.00,-25.0,44.00,50.00,False,155,2025-05-22 19:33:39,435.0,459.0,1966.0,64.586071,2.399898,2.399898,45.090909,19.200000,False,True,2025-05,2020-01
3,9,1,Windsong Annual Plan Comp,NaN,3,NaN,BMW,TX,2020-01-13 06:04:59,2024-03-18 22:06:42,41.0,2242.01,-25.0,83.00,25.00,False,172,2024-03-06 17:59:14,877.0,865.0,1556.0,51.116951,3.364833,1.121611,54.683171,13.034942,False,True,2024-04,2020-01
4,10,1,Ceramic Wash Monthly Membership Plan,50.00,3,NaN,Nissan,TX,2020-01-28 06:31:34,2025-08-08 07:07:00,67.0,4361.00,0.0,83.00,50.00,False,228,2025-08-04 21:32:38,361.0,357.0,2049.0,67.312746,3.387174,1.129058,65.089552,19.127193,False,True,2025-09,2020-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20272,28853,1,Good Wash Monthly Membership Plan,30.00,1,NaN,Toyota,TX,2026-07-31 22:10:07,2026-07-31 22:10:07,1.0,12.00,0.0,12.00,12.00,True,0,NaT,NaN,0.0,30.0,0.985545,0.000000,0.000000,12.000000,12.000000,True,False,NaN,2026-07
20273,28854,4,Better Wash Monthly Membership Plan,35.00,1,NaN,Chevrolet,TX,2026-07-31 22:15:05,2026-07-31 22:15:05,1.0,17.00,0.0,17.00,17.00,True,0,NaT,NaN,0.0,30.0,0.985545,0.000000,0.000000,17.000000,17.000000,True,False,NaN,2026-07
20274,28855,1,Good Wash Monthly Membership Plan,30.00,1,NaN,Honda,TX,2026-07-31 22:23:33,2026-07-31 22:23:33,1.0,12.00,0.0,12.00,12.00,True,0,NaT,NaN,0.0,30.0,0.985545,0.000000,0.000000,12.000000,12.000000,True,False,NaN,2026-07
20275,28856,3,Ceramic Wash Monthly Membership Plan,50.00,1,Passenger Vehicle,Ford,TX,2026-07-31 22:30:27,2026-07-31 22:30:27,1.0,25.00,0.0,25.00,25.00,True,0,NaT,NaN,0.0,30.0,0.985545,0.000000,0.000000,25.000000,25.000000,True,False,NaN,2026-07


---
## 8. Unit economics and CLV

Contribution = ARPU - (washes/month x variable cost per wash). CLV = monthly contribution / monthly
churn, the standard geometric-series lifetime.

CLV = monthly_contribution × expected_lifetime_months, and monthly_contribution = arpu - (washes_per_month × $4). So CLV only goes negative when a customer's monthly pay is less than what their actual wash usage costs to service.

**The variable cost per wash is an assumption, not data** — this export, like V1's, has no cost
side. \$4 (water, chemicals, power, incremental labour) is a plausible express-tunnel figure,
carried over as a starting point; the sensitivity check below shows how much rides on it.

In [42]:
if _seg_ok:
    churn_m = P.observed_monthly_churn(cust)
    ue = P.unit_economics(seg, variable_cost_per_wash=4)
    econ = ue.groupby("segment").agg(
        members=("customer_id", "size"), arpu=("arpu", "median"),
        wash_cost=("monthly_wash_cost", "median"),
        contribution=("monthly_contribution", "median"), clv=("clv", "median")).reindex(order)
    # print(f"observed monthly churn {churn_m:.2%}  ->  implied average lifetime {1 / max(churn_m, 1e-9):.1f} months")

    fig = go.Figure(go.Bar(
        x=econ.index, y=econ.clv, marker_color=[SEG_COLOR.get(s, T.s1) for s in econ.index],
        marker_line=dict(width=2, color=T.surface),
        text=[f"${v:,.0f}" for v in econ.clv], textposition="outside", textfont=dict(color=T.ink2),
        customdata=np.stack([econ.arpu, econ.contribution], axis=-1),
        hovertemplate="%{x}<br>CLV $%{y:,.0f}<br>ARPU $%{customdata[0]:.0f}, "
                      "contribution $%{customdata[1]:.0f}/mo<extra></extra>"))
    viz.style(fig, T, height=360, showlegend=False,
              title=dict(text="Median CLV per persona (at $4 variable cost per wash)"),
              yaxis=dict(title="CLV ($)"), xaxis=dict(title=""))
    fig.show()
    econ.round(2)
else:
    print("No segments to price -- see §7.")

In [43]:
# Same numbers as the bar chart above, but as a distribution rather than one median point per
# persona -- `ue` has one CLV value per CUSTOMER (not per persona), so this shows the full spread.
# box_visible adds the median/quartile box inline, since "median" is exactly what the bar chart
# collapsed each persona down to. points="all" shows every customer as a jittered dot with a rich
# hover -- at a few hundred-thousand customers per persona this stays responsive; if a much larger
# export ever makes it sluggish, drop to points="outliers" first.
if _seg_ok:
    fig = go.Figure()
    for s in order:
        sub = ue[ue.segment == s]
        if len(sub):
            fig.add_trace(go.Violin(
                x=[s] * len(sub), y=sub.clv, name=s,
                line_color=SEG_COLOR.get(s, T.s1), fillcolor=SEG_COLOR.get(s, T.s1), opacity=0.5,
                box_visible=True, points="all", pointpos=0, jitter=0.5,
                marker=dict(size=3, opacity=0.35, color=SEG_COLOR.get(s, T.s1)),
                showlegend=False,
                customdata=np.stack([sub.customer_id, sub.arpu, sub.washes_per_month,
                                     sub.monthly_contribution, sub.tenure_months,
                                     sub.package.fillna("--")], axis=-1),
                hovertemplate=(f"{s} · customer %{{customdata[0]}}<br>"
                              "CLV $%{y:,.0f}<br>"
                              "ARPU $%{customdata[1]:.2f}/mo · %{customdata[2]:.1f} washes/mo<br>"
                              "contribution $%{customdata[3]:.2f}/mo<br>"
                              "%{customdata[4]:.1f} mo tenure · %{customdata[5]}"
                              "<extra></extra>")))
    viz.style(fig, T, height=460, showlegend=False,
              title=dict(text="CLV distribution per persona (at $2.25 variable cost per wash)"),
              yaxis=dict(title="CLV ($)"), xaxis=dict(title=""))
    fig.add_hline(y=0, line_color=T.axis, line_width=1)
    fig.show()
else:
    print("No segments to price -- see §7.")

In [44]:
# Checks the mechanism behind any negative-CLV tail, rather than assuming it: contribution is
# negative exactly when arpu < washes_per_month x $2.25, i.e. the member's wash usage costs more
# to service than they pay. This compares negative-CLV members against their OWN persona's median
# on both arpu and washes_per_month, to see whether it's low pay, high usage, or both driving it.
if _seg_ok:
    bullets = []
    for s in order:
        sub = ue[ue.segment == s]
        neg = sub[sub.clv < 0]
        if len(sub) == 0:
            continue
        if len(neg) == 0:
            bullets.append(f"**{s}: no members with negative CLV** ({len(sub)} total).")
            continue
        _pct = len(neg) / len(sub)
        _arpu_ratio = neg.arpu.median() / sub.arpu.median() if sub.arpu.median() else float("nan")
        _wash_ratio = neg.washes_per_month.median() / sub.washes_per_month.median() if sub.washes_per_month.median() else float("nan")
        bullets.append(
            f"**{s}: {len(neg)} of {len(sub)} members ({_pct:.1%}) have negative CLV.** Among "
            f"them, median ARPU is {_arpu_ratio:.1f}x the persona's overall median and median "
            f"washes/month is {_wash_ratio:.1f}x the persona's overall median -- "
            + ("**lower pay** is the bigger factor" if _arpu_ratio < 1 and _wash_ratio >= 0.9 else
               "**higher wash usage** is the bigger factor" if _wash_ratio > 1 and _arpu_ratio >= 0.9 else
               "**both lower pay and higher usage** contribute" if _arpu_ratio < 1 and _wash_ratio > 1 else
               "neither alone explains it cleanly -- worth a closer look at this group specifically")
            + "."
        )
    insight(bullets if bullets else ["**No segments to check.**"], title="What's driving the negative tail")
else:
    print("No segments to price -- see §7.")

**What's driving the negative tail**

- **Power household: 56 of 2480 members (2.3%) have negative CLV.** Among them, median ARPU is 0.9x the persona's overall median and median washes/month is 3.8x the persona's overall median -- **lower pay** is the bigger factor.
- **Core regular: 102 of 4874 members (2.1%) have negative CLV.** Among them, median ARPU is 0.8x the persona's overall median and median washes/month is 3.9x the persona's overall median -- **lower pay** is the bigger factor.
- **Never activated: no members with negative CLV** (5869 total).
- **Promo flipper: 1029 of 6449 members (16.0%) have negative CLV.** Among them, median ARPU is 0.4x the persona's overall median and median washes/month is 1.9x the persona's overall median -- **lower pay** is the bigger factor.

In [45]:
if _seg_ok:
    _best_clv = econ.clv.idxmax()
    _worst_clv = econ.clv.idxmin()
    _ratio = econ.clv.max() / max(econ.clv.min(), 1e-9)
    insight([
        f"**Reading — `{_best_clv}` is worth {_ratio:.1f}x `{_worst_clv}`** by median CLV "
        f"(${econ.clv.max():,.0f} vs ${econ.clv.min():,.0f}).",
        "*Caveat:* CLV here uses one book-wide monthly churn rate rather than a per-persona rate, "
        "same simplification as V1 -- the true spread across personas is understated.",
    ])
else:
    print("No segments to price -- see §7.")

if _seg_ok:
    sens = pd.DataFrame([
        {"$/wash": vc,
         "unprofitable members": int((P.unit_economics(seg, vc).monthly_contribution < 0).sum()),
         "median CLV": P.unit_economics(seg, vc).clv.median(),
         "book contribution/mo": P.unit_economics(seg, vc).query("active").monthly_contribution.sum()}
        for vc in [1.00, 1.50, 2.25, 3.00, 4.00, 5.00]
    ])
    fig = go.Figure()
    fig.add_scatter(x=sens["$/wash"], y=sens["book contribution/mo"], mode="lines+markers",
                    line=dict(color=T.s1, width=2), marker=dict(size=9), showlegend=False,
                    hovertemplate="$%{x:.2f}/wash<br>$%{y:,.0f}/mo contribution<extra></extra>")
    viz.style(fig, T, height=340, title=dict(text="Active-book monthly contribution vs the cost assumption"),
              yaxis=dict(title="contribution ($/month)"), xaxis=dict(title="variable cost per wash ($)"))
    fig.show()

    _sign_changes = (sens["book contribution/mo"] > 0).nunique() > 1
    insight([
        (f"**Reading — book contribution stays {'positive' if (sens['book contribution/mo'] > 0).all() else 'mixed'} "
         f"across \\$1-\\$5/wash**, moving from \\${sens['book contribution/mo'].iloc[0]:,.0f} to "
         f"\\${sens['book contribution/mo'].iloc[-1]:,.0f}."),
        (f"**So-what — the conclusion {'is robust to' if not _sign_changes else 'DEPENDS ON'} the exact "
         "cost assumption** -- " + ("no sign change across the range tried." if not _sign_changes else
         "contribution crosses zero somewhere in \\$1-\\$5, so pin down the real marginal cost before acting.")),
    ])
    sens.round(2)


**Insights**

- **Reading — `Power household` is worth 2.2x `Promo flipper`** by median CLV ($1,057 vs $483).
- *Caveat:* CLV here uses one book-wide monthly churn rate rather than a per-persona rate, same simplification as V1 -- the true spread across personas is understated.

**Insights**

- **Reading — book contribution stays positive across \$1-\$5/wash**, moving from \$305,163 to \$202,949.
- **So-what — the conclusion is robust to the exact cost assumption** -- no sign change across the range tried.

---
## 9. Promo flippers — are discounts to this cluster worth it?

Two follow-up asks, both about the `Promo flipper` persona (yellow in every persona chart above):

1. **Value mismatch.** The hypothesis: the *good* core base is either (a) discount-acquired members
   who go on to pay normally cycle after cycle, or (b) recurring members who wash enough to
   actually use what they're paying for. The *bad* pattern is a member whose vehicle doesn't
   suggest they need a premium package -- an economy car on an expensive tier -- taking a deep
   signup discount and never washing enough to justify it: a discount that bought a signup, not a
   customer. §9a checks whether that pattern concentrates in `Promo flipper` more than in the
   other three personas.
2. **Recency + discount-conditioned win-back.** Does the discount depth on a member's LAST charge
   before they go quiet predict whether they come back -- mined from every historical lapse in the
   book (lapse-then-return, and lapse-with-no-return-yet), not assumed. §9b answers that, then
   ranks currently-quiet members by how recently they dropped, since a fresh lapse is a warmer
   win-back target than a stale one.

Both reuse `cust` / `seg` / `ev` from §7-§8 above; nothing here refits the clustering.

In [46]:
if _seg_ok:
    mix = seg.copy()
    mix["car_tier"] = P.car_value_tier(mix["vehicle_make"])
    mix["signup_discount_pct"] = (1 - mix["signup_amount"] / mix["list_price"]).clip(lower=0)

    tier_share = (mix.groupby(["segment", "car_tier"], observed=True).size()
                     .rename("members").reset_index())
    tier_share["share_within_segment"] = (
        tier_share["members"] / tier_share.groupby("segment")["members"].transform("sum"))

    econ_by_tier = (mix.groupby(["segment", "car_tier"], observed=True)
                       .agg(members=("customer_id", "size"),
                            avg_list_price=("list_price", "mean"),
                            avg_signup_discount=("signup_discount_pct", "mean"),
                            churn_rate=("churned", "mean"))
                       .reset_index())
    focus = econ_by_tier[econ_by_tier.segment == "Promo flipper"].sort_values("members", ascending=False)
    focus.round(2)
else:
    print("No segments to check -- see §7.")

In [47]:
if _seg_ok:
    tier_order = [t for t in ["Economy", "Mid", "Premium", "Unknown"] if t in tier_share.car_tier.unique()]
    tier_color = {"Economy": T.good, "Mid": T.s1, "Premium": T.critical, "Unknown": T.muted}
    order_labels9 = [f"{s} (C{int(prof.loc[s, 'segment_id'])})" for s in order]
    pivot = (tier_share.pivot(index="segment", columns="car_tier", values="share_within_segment")
                       .reindex(order)[tier_order])

    fig = go.Figure()
    for tier in tier_order:
        fig.add_bar(x=order_labels9, y=pivot[tier], name=tier, marker_color=tier_color.get(tier, T.muted),
                    marker_line=dict(width=1, color=T.surface),
                    hovertemplate="%{x}<br>" + tier + " %{y:.0%}<extra></extra>")
    viz.style(fig, T, height=380, barmode="stack",
              title=dict(text="Vehicle value tier mix per persona"),
              yaxis=dict(title="share of members", tickformat=".0%"), xaxis=dict(title=""))
    fig.show()
else:
    print("No segments to chart -- see §7.")

In [48]:
if _seg_ok:
    _pf_tier = tier_share[tier_share.segment == "Promo flipper"].set_index("car_tier")["share_within_segment"]
    _other_tier = tier_share[tier_share.segment != "Promo flipper"].groupby("car_tier")["members"].sum()
    _other_tier = _other_tier / _other_tier.sum()
    _econ_pf = _pf_tier.get("Economy", 0.0)
    _econ_other = _other_tier.get("Economy", 0.0)
    _pf_discount = mix.loc[mix.segment == "Promo flipper", "signup_discount_pct"].mean()
    _other_discount = mix.loc[mix.segment != "Promo flipper", "signup_discount_pct"].mean()
    _mismatch = (_econ_pf > _econ_other) and (_pf_discount > _other_discount)
    insight([
        f"**Reading — Economy-tier vehicles are {_econ_pf:.0%} of `Promo flipper` members** vs "
        f"{_econ_other:.0%} across the other three personas.",
        f"**Reading — `Promo flipper`'s average signup discount is {_pf_discount:.0%}** vs "
        f"{_other_discount:.0%} elsewhere in the book.",
        (f"**So-what — the mismatch pattern (economy car, deep discount) is "
         f"{'more' if _mismatch else 'not clearly more'} concentrated in `Promo flipper`** than in "
         "the rest of the book -- " +
         ("consistent with the hypothesis that this cluster is acquired on price rather than value."
          if _mismatch else
          "this export doesn't cleanly support the 'cheap car, premium package' story on its own; "
          "the discount/reactivation evidence in §9b below is the sharper test.")),
        "*Caveat:* `car_tier` is a static make->tier lookup (`P.CAR_VALUE_TIER`), not a real "
        "vehicle valuation -- read the split as directional, and `Unknown` makes are excluded from "
        "both shares above rather than folded into either bucket.",
    ])
else:
    insight(["**No segments to read yet.**"])

**Insights**

- **Reading — Economy-tier vehicles are 65% of `Promo flipper` members** vs 66% across the other three personas.
- **Reading — `Promo flipper`'s average signup discount is 35%** vs 28% elsewhere in the book.
- **So-what — the mismatch pattern (economy car, deep discount) is not clearly more concentrated in `Promo flipper`** than in the rest of the book -- this export doesn't cleanly support the 'cheap car, premium package' story on its own; the discount/reactivation evidence in §9b below is the sharper test.
- *Caveat:* `car_tier` is a static make->tier lookup (`P.CAR_VALUE_TIER`), not a real vehicle valuation -- read the split as directional, and `Unknown` makes are excluded from both shares above rather than folded into either bucket.

**§9b — discount-conditioned win-back, mined from every historical lapse**

`P.lapse_discount_reactivation(ev, churn_after=CHURN_AFTER_V2)` walks every charge in the book and
flags it as a **lapse** if the customer then went quiet for more than 90 days (`CHURN_AFTER_V2`) --
whether they eventually paid again (a resolved lapse) or are still quiet as of the export's last day
(an open lapse, same censoring convention used everywhere else in this notebook). Each lapse carries
the discount depth of the charge that preceded the silence, bucketed `None` / `Light (<15%)` /
`Moderate (15-30%)` / `Deep (30%+)` -- this is the mirror image of `winback_events` in §5, which
looks at whether the RETURN charge was discounted rather than the charge before the silence.

In [49]:
lapse = P.lapse_discount_reactivation(ev, churn_after=CHURN_AFTER_V2)
summ = P.lapse_discount_summary(lapse)
summ.round(3)

,n_lapses,reactivation_rate,avg_days_to_return
discount_bucket,,,
None,7861,0.158,241.149
Light (<15%),3094,0.185,360.805
Moderate (15-30%),1114,0.197,332.292
Deep (30%+),1716,0.110,310.867


In [50]:
if _seg_ok:
    lapse_seg = lapse.merge(seg[["customer_id", "segment"]], on="customer_id", how="left")
    lapse_seg = lapse_seg[lapse_seg.segment.notna()]
    bucket_order = ["None", "Light (<15%)", "Moderate (15-30%)", "Deep (30%+)"]

    cmp = (lapse_seg.assign(is_pf=lapse_seg.segment.eq("Promo flipper"))
                    .groupby(["is_pf", "discount_bucket"], observed=True)
                    .agg(n=("returned", "size"), reactivation_rate=("returned", "mean"))
                    .reset_index())

    fig = go.Figure()
    for is_pf, label, color in [(True, "Promo flipper", SEG_COLOR["Promo flipper"]),
                                (False, "Other 3 personas", T.muted)]:
        sub = cmp[cmp.is_pf == is_pf].set_index("discount_bucket").reindex(bucket_order)
        fig.add_bar(x=bucket_order, y=sub.reactivation_rate, name=label, marker_color=color,
                    marker_line=dict(width=1, color=T.surface),
                    text=[f"n={int(v)}" if pd.notna(v) else "n=0" for v in sub.n],
                    textposition="outside",
                    hovertemplate="%{x}<br>%{y:.0%} reactivated<br>%{text}<extra></extra>")
    viz.style(fig, T, height=400, barmode="group",
              title=dict(text="Reactivation rate by discount depth at the last charge before going quiet"),
              yaxis=dict(title="reactivation rate", tickformat=".0%"), xaxis=dict(title=""))
    fig.show()
else:
    print("No segments to split by -- see §7.")

In [51]:
if _seg_ok:
    bucket_order = ["None", "Light (<15%)", "Moderate (15-30%)", "Deep (30%+)"]
    _pf_rate = cmp[cmp.is_pf].set_index("discount_bucket")["reactivation_rate"].reindex(bucket_order)
    _other_rate = cmp[~cmp.is_pf].set_index("discount_bucket")["reactivation_rate"].reindex(bucket_order)
    _pf_deep, _pf_none = _pf_rate.get("Deep (30%+)"), _pf_rate.get("None")
    _have_both = pd.notna(_pf_deep) and pd.notna(_pf_none)
    _lifts_pf = _have_both and _pf_deep > _pf_none

    _book_line = ", ".join(
        f"{b} {summ.loc[b, 'reactivation_rate']:.0%} (n={int(summ.loc[b, 'n_lapses'])})"
        for b in bucket_order if b in summ.index)
    insight([
        f"**Reading — book-wide reactivation by discount bucket at the last charge:** {_book_line}.",
        (f"**So-what — for `Promo flipper`, a deep discount at the last charge "
         f"{'does' if _lifts_pf else 'does NOT clearly'} raise reactivation** "
         f"(None {_pf_none:.0%} -> Deep {_pf_deep:.0%})"
         if _have_both else
         "**So-what — not enough `Promo flipper` lapses in one or more buckets to compare cleanly.**"),
        "*Caveat:* correlation, not a controlled experiment -- discounts weren't randomly assigned, "
        "so a bucket's higher reactivation rate may reflect who tends to get offered a deep "
        "discount (e.g. higher-value members) rather than the discount itself causing the return.",
    ])
else:
    insight(["**No segments to read yet.**"])

**Insights**

- **Reading — book-wide reactivation by discount bucket at the last charge:** None 16% (n=7861), Light (<15%) 19% (n=3094), Moderate (15-30%) 20% (n=1114), Deep (30%+) 11% (n=1716).
- **So-what — for `Promo flipper`, a deep discount at the last charge does NOT clearly raise reactivation** (None 9% -> Deep 6%)
- *Caveat:* correlation, not a controlled experiment -- discounts weren't randomly assigned, so a bucket's higher reactivation rate may reflect who tends to get offered a deep discount (e.g. higher-value members) rather than the discount itself causing the return.

In [52]:
if _seg_ok:
    open_lapses = (lapse[~lapse.returned]
                   .merge(cust[["customer_id", "vehicle_make", "package", "arpu", "days_since_wash"]],
                          on="customer_id", how="left")
                   .merge(seg[["customer_id", "segment"]], on="customer_id", how="left"))
    open_lapses["car_tier"] = P.car_value_tier(open_lapses["vehicle_make"])
    open_lapses = open_lapses.merge(
        summ["reactivation_rate"].rename("historical_reactivation_rate"),
        left_on="discount_bucket", right_index=True, how="left")

    candidates = (open_lapses.sort_values("days_quiet")
                             [["customer_id", "segment", "package", "car_tier", "discount_bucket",
                               "days_quiet", "arpu", "historical_reactivation_rate"]]
                             .reset_index(drop=True))
    candidates.head(20)
else:
    print("No segments to prioritize -- see §7.")

In [53]:
if _seg_ok:
    _n_fresh = int((candidates.days_quiet <= 30).sum())
    _pf_in_candidates = candidates.segment.eq("Promo flipper").mean() if len(candidates) else np.nan
    _pf_book_share = seg.segment.eq("Promo flipper").mean()
    insight([
        f"**Reading — {len(candidates):,} members are currently quiet** (no charge in the last "
        f"{CHURN_AFTER_V2} days, never returned since); {_n_fresh:,} of them dropped within the "
        "last 30 days -- the warmest win-back targets by the recency framing above.",
        (f"**So-what — `Promo flipper` is {_pf_in_candidates:.0%} of the currently-quiet list** vs "
         f"{_pf_book_share:.0%} of the clustered book" if len(candidates) else
         "**No currently-open lapses in this export.**"),
        "Each row's `historical_reactivation_rate` is the book-wide rate for that member's own "
        "discount bucket at their last charge (from §9b above) -- a prioritization signal, not a "
        "per-member prediction.",
    ])
else:
    insight(["**No segments to read yet.**"])

**Insights**

- **Reading — 11,594 members are currently quiet** (no charge in the last 90 days, never returned since); 0 of them dropped within the last 30 days -- the warmest win-back targets by the recency framing above.
- **So-what — `Promo flipper` is 54% of the currently-quiet list** vs 33% of the clustered book
- Each row's `historical_reactivation_rate` is the book-wide rate for that member's own discount bucket at their last charge (from §9b above) -- a prioritization signal, not a per-member prediction.

---
## Reproducing this

```bash
conda activate sonnys
jupyter lab experiments/customer-profiling/customer_profilling_V2.ipynb
streamlit run experiments/customer-profiling/app.py   # V1's demo -- not yet wired to this export
```

Shared logic lives in [`profiling.py`](profiling.py), the same module V1 and `app.py` import — this
notebook only differs from V1 in which file it loads (`DATA_V2`, a `.parquet`, via
`P.load_events(DATA_V2, drop_cols=[])`) and in generating every "Insights" block live from that run's
own numbers rather than restating numbers from a prior look at the data.